In this notebook I want to run Bayesian regression models using Bambi: 

In [ ]:
# %matplotlib inline
# %reload_ext autoreload
# %autoreload 2

In [ ]:
import arviz as az
import bambi as bmb
import argparse
from os.path import join
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import zscore, spearmanr, pearsonr
import os
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns
import xarray as xr
from matplotlib.backends.backend_pdf import PdfPages


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
import sys
sys.path.append('/hpc/users/qasims01/resources/LFPAnalysis')

## set some plotting parameters

In [ ]:
# Set some plot defaults for matplotlib (when not using seaborn): 

import  matplotlib.font_manager

path = '/sc/arion/projects/guLab/Salman/Arial.ttf'
matplotlib.font_manager.fontManager.addfont(path)

prop = matplotlib.font_manager.FontProperties(fname=path)

# Set the default font family to the registered custom font
plt.rcParams['font.family'] = prop.get_name()

# Rest of your customization code
plt.rcParams['font.size'] = 20
plt.rcParams['lines.linewidth'] = 1

# Set palette: 

# Define RGB values for your two colors
ignacio_purple = np.array([86, 28, 70]) / 255
ignacio_blue = np.array([0, 70, 126]) / 255 

# Create a Seaborn palette with your custom colors normalized to 0-1
custom_palette = sns.color_palette([ignacio_purple, ignacio_blue])

ignacio_orange = np.array([255, 172, 77]) / 255
ignacio_mauve = np.array([193, 155, 211]) / 255
ignacio_beige = np.array([205, 182, 162]) / 255

# Set the custom palette
sns.set_palette(custom_palette)
sns.set_context("paper")

# from YBA
# from YBA
hpc_c = np.array([0.44314, 0.2902, 1])
amy_c = np.array([0.49412, 0, 0.90196])
ofc_c = np.array([1, 0.75686, 0.41569])
acc_c = np.array([0.7098, 0.27059 ,0.68235])
dmpfc_c = np.array([0.40784, 0.01961, 0])
dlpfc_c = np.array([1, 0.06275, 0.06275])

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/sc/arion' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

The following function sets up your regression formula, and allows you to determine how to save the results and plots: 

In [ ]:
# import nutpie

def make_formula(y=None, X=None, Intx=None, rand_effect=None, rand_slopes_group=None, rand_slopes_predictor=None, categorical=None, categorical_baseline=True, rand_intx=False):
    if rand_effect is not None:
        # mixed/hierarchial/multilevel modeling:
        if categorical_baseline: # this means we set a baseline for categorical effects 
            rand_term = [f'(1|{x})' for x in rand_effect]
            if rand_slopes_group is not None: # do we want a more complex mixed-effects model with random slopes as well as intercepts?
                rand_slope_terms = [f'(1+'+'+'.join(rand_slopes_predictor)+f'|{b})' for b in rand_slopes_group]
                # [f'({a}|{b})' for a in rand_slopes_predictor for b in rand_slopes_group]
                if not Intx:
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)
                else:
                    if rand_intx:
                        rand_slope_terms_Intx = [f'(1+'+'+'.join(Intx)+f'|{b})' for b in rand_slopes_group]
                        formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)+'+'+'+'.join(rand_slope_terms_Intx)
                    else:
                        formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)
            else:
                if not Intx: 
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)
                else:
                    formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)
        else: # this means all levels of categorical effect are judged absent or present 
             # format random effects properly 
            rand_term = [f'(0+1|{x})' for x in rand_effect]
            if rand_slopes_group is not None: 
                rand_slope_terms = [f'(0+'+'+'.join(rand_slopes_predictor)+f'|{b})' for b in rand_slopes_group]
                if not Intx:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)
                else:
                    rand_slope_terms_Intx = [f'(0+'+'+'.join(Intx)+f'|{b})' for b in rand_slopes_group]
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(rand_slope_terms)+'+'+'+'.join(Intx)+'+'+'+'.join(rand_slope_terms_Intx)
            else:
                if not Intx:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)
                else:
                    formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)   

    return formula
    
    
def run_model(df, formula=None, categorical=None,
              priors=None, cores=2, chains=2, tune=1500, draws=2000, target_accept=0.95, model_fam='bernoulli', 
              output_dir='/sc/arion/work/qasims01/MemoryBanditData/EMU/BayesModels', return_model=False, save_model_res = True, label=None,
             sampler='numpyro_nuts'):
    """
    Run the model: 
    
    Parameters
    ----------
    df: pandas dataframe 
        All the variables 
    y: str
        Name of the dependent variable
    X: list of strings 
        Names of the independent variables 
    Intx: list of strings
        Names of the interactions, formatted as 'a:b' 
    rand_effect: str
        Name of variable for random effects (i.e. 'subject') 
    rand_slopes: bool 
        Should we fit random slopes in addition to random intercepts? 
    priors: dict 
        Prior distribution for any independent variables (i.e. Intercept, random intercept) 
        Default to weakly informative priors (Gelman et al. 2008) 
    categorical: list of strings 
        Names of categorical variables going into model 
    
    Returns
    -------
    results: model
    
    """

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # if no label provided, make your own
    if not label: 
        label = (f"{y}" + "_{}"*len(X)).format(*X)
    
        
    # construct the model 
    model = bmb.Model(formula=formula, 
              data=df,
             family=model_fam,
             priors=priors,
             categorical=categorical)
    
    # fit the model 
    if sampler == 'nutpie':
        compiled_model = nutpie.compile_pymc_model(model)
        results = nutpie.sample(compiled_model)
    else:
        results=model.fit(cores=cores, 
                              chains=chains, 
                              tune=tune, 
                              draws=draws, 
                              target_accept=target_accept,
                             inference_method=sampler,
                             idata_kwargs={"log_likelihood": True})
    # ,init="advi+adapt_diag"

    # save out the CSV with the results 
    summary = az.summary(results, hdi_prob=0.95)
    for var in summary.index.values:
        if var.endswith("]"):  # skip individual subject parameters
            continue
        if var.startswith("1"):  # skip random effects.
            continue
        summary.loc[var, "P>0"] = np.mean(results.posterior[var].values > 0)
    summary.to_csv(join(output_dir , f"{label}_summary.csv"))

    
    if save_model_res:
        # save out the model for model comparison (THIS WILL BE GB-sized, so really only needed for model comparison)
        az.to_netcdf(results, join(output_dir , f"{label}_model"))
    
    if return_model:
        return model, results 
    
    
def plot_res(fitted_model, X, ax=None, save_label=None): 
    """
    Code for plotting select beta weights in a publication friendly manner from saved csv
    """

    axes = az.plot_forest(fitted_model,
                           kind='ridgeplot',
                           var_names=[f'^(?!.*\|).*{x}.*$' for x in X],
                           filter_vars="regex",
                           colors='gray',
                           combined=True,
                           ridgeplot_overlap=2,
                           ridgeplot_truncate=False,
                           ridgeplot_alpha=0.4,
                           hdi_prob=0.95,
                           ax=ax)
    # 'forestplot'
    ax.vlines(0, ax.get_ylim()[0], ax.get_ylim()[1], color = 'black')
    # plt.show()
#     plt.savefig(save_label, dpi=300)
#     plt.close()

def print_latex_table(fitted_model, X):
    """
    Code for printing the pub-friendly table of model details from model csv table
    """
    summary = az.summary(fitted_model, hdi_prob=0.95)
    
    # print out the latex table for the cell output
    results_df = summary.loc[:,
                   ['mean', 'sd', 'hdi_2.5%', 'hdi_97.5%']
                  ]
    all_tests = [] 
    for x in X: 
        temp = results_df.filter(like=f'{x}', axis=0)
        all_tests.append(temp)

    results_df = pd.concat(all_tests)

    c_string = '|c'*results_df.shape[0] + '|'
    print(results_df.reset_index().to_latex(index=False, 
                                            column_format=c_string).replace("\\\n", "\\ \hline\n"))  
    
def plot_predictions(df, model, fitted_model, X, y='recalled',  
                    save_dir='/home1/salman.qasim/Salman_Project/FR_Emotion/BayesModels'):
    """
    Code for plotting the model predictions 
    """
    
    # sample from posterior-predictive distribution to make in-sample predictions 
    model.predict(fitted_model, kind="pps")
    y_posterior = fitted_model.posterior[f"{y}_mean"].stack(samples=("chain", "draw")).values
    # Select 25% of the values in the posterior, making sure we take values from both chains.
    y_posterior = recall_posterior[:, ::4]

    for x in X:
        save_file = join(save_dir, f'{x}_PredictRecallplot.pdf')
        with PdfPages(save_file) as pdf:
            f, ax = plt.subplots(1, 1, figsize=[3,3], dpi=300)
            for pps in y_posterior.T:
                sns.regplot(data=df, x=x, y=pps, ci=None, scatter=False, line_kws={'alpha':0.1}, color='k')
            sns.regplot(data=df, x=x, y=y_posterior.mean(axis=1), ci=None, scatter=False, color='r')
            pdf.savefig()
            plt.close()

Prepare the model data

Now, let's set up our predictors and formulas: 

Update (6/6/2024):

If I'm running my big mixed-effects models and then comparing across region, I shouldn't split by region before running? Checking this now to be sure. 

## Detect RPE sensitivity during the encoding task

In [ ]:
# LOAD the data
analysis_evs = ['feedback_start']
chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['urpe'] = np.abs(chan_df.rpe)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

            memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials'])

            elec_counter +=1

            chan_dfs.append(chan_df)

chan_dfs = pd.concat(chan_dfs)
chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))

## Compare variance across frequency bands:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming your dataframe is named 'df'
# Group by 'participant', 'band', and 'channel' and compute variance of 'tfr'
variance_df = chan_dfs.groupby(['participant', 'fband', 'channel'])['tfr'].var().reset_index()

# Rename the computed variance column
variance_df.rename(columns={'tfr': 'tfr_variance'}, inplace=True)


In [ ]:
np.nanmedian(variance_df[variance_df.fband=='hfa'].tfr_variance)

In [ ]:
variance_df[variance_df.tfr_variance>5]

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='fband', y='tfr_variance', data=variance_df)

# Add plot labels
plt.title('Distribution of TFR Variances Across Channels by Band and Participant')
plt.xlabel('Band')
plt.ylabel('TFR Variance')

# Show plot
plt.show()

In [ ]:
# chan_dfs[np.abs(chan_dfs.rpe)<=0.0001]

In [ ]:
# chan_dfs[np.isnan(chan_dfs.tfr)].tfr

## full data bayesian GLM approach: 

First, I want to model all of the data together in a big boy model that includes frontal cortex and medial temporal lobe regions with enough electrodes in them/

In [ ]:
import re

def extract_number(string):
    match = re.search(r'\[(\d+(?:\.\d+)?)\]', string)
    return int(float(match.group(1))) if match else None

In [ ]:
import gc
from joblib import Parallel, delayed

# TRUE MEM REGRESSION
analysis_evs = 'feedback_start'
rois = ['AMY', 'dlPFC', 'dmPFC', 'OFC', 'HPC', 'ACC']
bands = ['theta', 'hfa'] 
model_results = []
df_loos = []
chan_dfs_list = []

# Load subject data once outside the loop
subj_data = {subj_id: pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df') 
             for subj_id in subj_ids}

# Preprocessing subject-level data and demographics
elec_counter = 0 
for subj_id, elec_df in subj_data.items():
    for chan in elec_df.label.unique():
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv')
        
        chan_df['unique_elec_id'] = elec_counter
        chan_df['urpe'] = np.abs(chan_df.rpe)
        chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2 * np.nanstd(chan_df.rpe))
        chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2 * np.nanstd(chan_df.urpe))
        chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2 * np.nanstd(chan_df.DPRIME))
        chan_df['age'] = subj_df[subj_df.MSSMCode == subj_id].Age.values[0]
        chan_df['sex'] = subj_df[subj_df.MSSMCode == subj_id].Sex.str.strip().values[0]
        chan_df['hemi'] = elec_df[elec_df.label == chan].hemisphere.str.lower().values[0]
        chan_df['zreward'] = chan_df.reward.apply(lambda x: -1 if x == 0 else x)

        elec_counter += 1
        chan_dfs_list.append(chan_df)

# Combine all channel DataFrames outside the loop
chan_dfs = pd.concat(chan_dfs_list)
chan_dfs['zAge'] = (chan_dfs.age - np.nanmean(chan_dfs.age)) / (2 * np.nanstd(chan_dfs.age))
# chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]

def run_band_model(time_averaged_df, rois, band):
    # Filter the data for the current frequency band
    
    time_averaged_df = time_averaged_df[time_averaged_df['region'].isin(rois)]
    
    model_df = time_averaged_df[(time_averaged_df.fband == band)].reset_index(drop=True).dropna()
    res_df = time_averaged_df[(time_averaged_df.fband == band)].reset_index(drop=True)[['participant', 'hemi', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    if len(rois) > 1:
        # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + hemi + region + (1|participant/region) + (0 + zrpe + zpm|participant/region) + (1|unique_elec_id) + (0 + zrpe + zpm|unique_elec_id)'
        # uncorrelated slopes and intercepts
        # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + hemi + region + (1|participant/region) + (0 + zrpe + zpm|participant/region) + (1|participant/unique_elec_id) + (0 + zrpe + zpm|participant/unique_elec_id)'
        categorical = ['region', 'hemi', 'participant', 'channel']
        label = f'{band}_{analysis_evs}_rpe_pm_expanded_regions'
    else: 
        # participant and hemi are crossed (same hemi appears in all participants), while participants and elec_id are nested
        formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1|participant/hemi) + (1 + zrpe + zpm|channel)'
        # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + hemi + (1|participant) + (1 + zrpe + zpm|unique_elec_id)'
        categorical = ['hemi', 'participant', 'channel']
        label = f'{band}_{analysis_evs}_{rois[0]}_rpe_pm_single_regions'
        res_df['region'] = rois[0]

    # Run the Bambi model
    model_rpe, results_rpe = run_model(model_df, formula=formula_zrpe, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, 
                               model_fam='gaussian', output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res=True, label=label, sampler='mcmc')
    
    result_df_summary = az.summary(results_rpe)
    rpe_values = result_df_summary[result_df_summary.index.str.startswith("zrpe|")]['mean'].values
    pm_values = result_df_summary[result_df_summary.index.str.startswith("zpm|")]['mean'].values

    elec_names = result_df_summary[result_df_summary.index.str.startswith("zrpe|")]['mean'].index
    elec_nums = [extract_number(x) for x in elec_names]

    for elec_ix, elec_id in enumerate(elec_nums):
        res_df.loc[res_df.unique_elec_id == elec_id, 'beta_rpe'] = result_df_summary[result_df_summary.index == "zrpe"]['mean'].values[0] + rpe_values[elec_ix]
        res_df.loc[res_df.unique_elec_id == elec_id, 'beta_pm'] = result_df_summary[result_df_summary.index == "zpm"]['mean'].values[0] + pm_values[elec_ix]

    roi = time_averaged_df.region.unique()[0]
    res_df['band'] = band 
    
    res_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{label}', index=False)
    
    del results_rpe
    del model_rpe
    gc.collect()

# Run across bands
for region in rois:
    for band in bands:
        print(f'running {band} now')
        run_band_model(chan_dfs, rois=[region], band=band)

In [ ]:
test_res = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/theta_feedback_start_AMY_rpe_pm_single_regions_model')



In [ ]:
del test_res

In [ ]:
az.plot_trace(test_res)
plt.tight_layout()

In [ ]:
# # TRUE MEM REGRESSION

# analysis_evs = 'feedback_start'
# rois = ['dmPFC', 'OFC', 'HPC', 'AMY', 'ACC', 'dlPFC']
# bands = ['hfa'] 
# model_results = []
# df_loos = []

# # Load subject data once outside the loop
# subj_data = {subj_id: pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df') 
#              for subj_id in subj_ids}


# for band in bands: 
#     # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
#     chan_dfs = [] 
#     glms = []
#     elec_counter = 0 
#     for subj_id in subj_ids:
#         print(f'Loading subj {subj_id}')
#         day = 'Day1'
#         filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

#         for chan in elec_df.label.unique():
#             # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
#             chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df_time0-1.5.csv')


#             chan_df['unique_elec_id'] = elec_counter
#             chan_df['urpe'] = np.abs(chan_df.rpe )
#             chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#             chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
#             chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
#             chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#             chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
#             chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
#             chan_df['zreward'] = chan_df.reward.apply(lambda x: -1 if x==0 else x)
        
#             elec_counter +=1

#             chan_dfs.append(chan_df)

#     chan_dfs = pd.concat(chan_dfs)
#     chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
        
#     # restrict to electrodes in rois: 
#     chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]

#     rel_col = ['tfr', 'zrpe', 'zpm', 'unique_elec_id', 'participant', 'region', 'hemi']
        
#     model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
        
#     res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'hemi', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#     # RPE second
#     # Crossed effects for region across participants, nested effects for electrode across region across participants 
#     # Reference: https://stats.stackexchange.com/questions/228800/crossed-vs-nested-random-effects-how-do-they-differ-and-how-are-they-specified
#     # Nested (this might be too extreme of an assumption (8/7/2024)
#     # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1|participant) + (1|region) + (1 + zrpe + zpm|unique_elec_id)'
        
#     # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1 + zrpe + zpm|participant) + (1 + zrpe + zpm|region) + (1 + zrpe + zpm|unique_elec_id)'
# #     formula_zrpe = """
# #     tfr ~ 1 + zrpe + zpm + hemi + zrpe * region + zpm * region +
# #     (1 + zrpe + zpm | unique_elec_id) +
# #     (1 | region)
# # """
#     #     (1 | participant) +
#     # Crossed
#     formula_zrpe = 'tfr ~ 1 + zrpe + zpm + hemi + region + (1|participant) + (1 + zrpe + zpm|unique_elec_id)'
#     categorical = ['region', 'hemi', 'participant', 'unique_elec_id'] 
#     label = f'{band}_{analysis_evs}_rpe_pm_expanded_regions'
#     model_rpe, results_rpe = run_model(model_df, formula=formula_zrpe, categorical=categorical,
#                                priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, model_fam='t', 
#                                output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
#                                return_model=True, save_model_res=True, label=label, sampler='mcmc')
        
        
#     result_df_summary = az.summary(results_rpe)
#     rpe_values = result_df_summary[result_df_summary.index.str.startswith("zrpe|")]['mean'].values
#     pm_values = result_df_summary[result_df_summary.index.str.startswith("zpm|")]['mean'].values

#     elec_names = result_df_summary[result_df_summary.index.str.startswith("zrpe|")]['mean'].index
#     elec_nums = [extract_number(x) for x in elec_names]

#     for elec_ix, elec_id in enumerate(elec_nums):
#         res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = result_df_summary[result_df_summary.index=="zrpe"]['mean'].values[0] + rpe_values[elec_ix]
#         res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = result_df_summary[result_df_summary.index=="zpm"]['mean'].values[0] + pm_values[elec_ix]

# #     for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
# #         res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = results_rpe['posterior']['zrpe'].mean().values + results_rpe['posterior']['zrpe|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
# #         res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results_rpe['posterior']['zpm'].mean().values + results_rpe['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

#     res_df['band'] = band 
    
#     res_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_{band}_rpe_pm_df_bayes_expanded_regions', index=False)

#     del results_rpe
#     del model_rpe
    
#     # model_results.append(res_df)

# # gamble_model_df = pd.concat(model_results)
# # gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_rpe_pm_expanded_regions', index=False)


In [ ]:
results_rpe = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/theta_feedback_start_rpe_pm_full_hierarchy_fixed_region_model')



In [ ]:
# Load the data after running:
analysis_evs = 'feedback_start'
band = 'beta'
gamble_model_df = [] 
for roi in ['dmPFC', 'dlPFC', 'AMY', 'OFC', 'HPC']:
    gamble_model_df.append(pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_feedback_start_{roi}_rpe_pm_single_regions'))
gamble_model_df = pd.concat(gamble_model_df).reset_index()

In [ ]:
# Make some plots

In [ ]:
az.plot_trace(results_rpe, var_names=['hemi'])

9/15/2024: 

When I residualize reward out of RPE, there is a 

1. big lateralized effect of R dmPFC, R OFC HFA. some right HPC

2. some R OFC theta (decrease)

3. big lateralize effect of R OFC beta 

In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(5, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_hierarchical.pdf') as pdf:
    sns.barplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe',errorbar='se', 
            order = ['dmPFC', 'dlPFC', 'OFC', 'ACC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
        order = ['dmPFC', 'dlPFC', 'OFC', 'ACC', 'HPC', 'AMY'], alpha=0.1, color='k')

    plt.hlines([0], -1, 6, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,6])
    pdf.savefig(fig)

However the regional results seem to replicated in the aggregate... ? 

In [ ]:
# is there a difference in the RPE sensitivity coding of different regions?

region_lm = smf.ols('beta_rpe ~ C(region)', data=gamble_model_df[gamble_model_df.band==band]).fit()

table = sm.stats.anova.anova_lm(region_lm, typ=2) # Type 2 ANOVA DataFrame
print(table)

First, let's determine if 'reward' vs. 'rpe' is a better predictor of neural activity:

Update 6/6/2024: Did I do this? What was the outcome for hfa? Maybe I did this for behavior, and not brain activity, and found that RPE explained behavior better. 

In [ ]:
# chan_dfs['p_reward'] = chan_dfs.apply(lambda x: 1 if x.reward==1 else -1, axis=1)

In [ ]:
# # REWARD REGRESSION

# rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
# bands = ['hfa'] 
# model_results = []
# for band in bands: 
#     for region in rois:
        
#         rel_col = ['tfr', 'p_reward', 'zpm', 'unique_elec_id', 'hemi']
#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
#         res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#         formula = 'tfr ~ 1 + p_reward + zpm + hemi + (1 + p_reward + zpm|unique_elec_id)'
#         categorical = ['hemi'] 
#         label = f'{region}_{band}_encoding_reward_pm_only_gaussian'
#         model, results = run_model(model_df, formula=formula, categorical=categorical,
#                                    priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
#                                    output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
#                                    return_model=True, save_model_res = True, label=label)

#         res_df['beta_reward'] = np.nan
#         for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = results['posterior']['p_reward'].mean().values + results['posterior']['p_reward|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
#         res_df['band'] = band 
#         model_results.append(res_df)
        
# gamble_model_df = pd.concat(model_results)
# gamble_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_reward_pm_only_gaussian_channel_df', index=False)


Now, do the same thing for SME effects at encoding

In [ ]:
# TRUE MEM REGRESSION

analysis_evs = 'feedback_start'
rois = ['OFC', 'ACC', 'dmPFC', 'HPC', 'AMY']  # , 'AINS', 'vlPFC', 'dlPFC'
bands = ['theta', 'alpha', 'beta', 'hfa'] 
model_results = []
df_loos = []
for band in bands: 
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df.csv')

            chan_df['unique_elec_id'] = elec_counter
            chan_df['urpe'] = np.abs(chan_df.rpe )
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['binarized_rpe'] = chan_df.rpe.apply(lambda x: 1 if x>0 else -1)
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            chan_df['zreward'] = chan_df.reward.apply(lambda x: -1 if x==0 else x)
            
            formula = 'zrpe ~ 1 + zreward'
            residual_model = smf.ols(formula=formula, data=chan_df).fit()
            rpe_residuals = residual_model.resid

            chan_df['rpe_resid'] = rpe_residuals
            
            # Add memory information                    
            memory_df = combined_df[(combined_df.condition=='Day1') & (combined_df.participant==chan_df.participant.unique()[0])].dropna(subset=['trials_gamble']).sort_values(by=['trials_gamble'])[['participant', 'trials_gamble', 'hits']]
            memory_df.rename(columns={'trials_gamble':'trials'}, inplace=True)
            memory_df['phit'] = memory_df.apply(lambda x: 1 if x.hits==True else -1, axis=1)
            chan_df = chan_df.merge(memory_df, on=['participant', 'trials']) 
    
            elec_counter +=1

            chan_dfs.append(chan_df)

    chan_dfs = pd.concat(chan_dfs)
    chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
    
    # restrict to electrodes in rois: 
    chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]             
            
    rel_col = ['tfr', 'phit', 'rpe_resid', 'zpm', 'unique_elec_id', 'participant', 'region', 'hemi']
        
    model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
        
    res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'hemi', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    # RPE second
    # Crossed effects for region across participants, nested effects for electrode across region across participants 
    # Reference: https://stats.stackexchange.com/questions/228800/crossed-vs-nested-random-effects-how-do-they-differ-and-how-are-they-specified
    # Nested (this might be too extreme of an assumption (8/7/2024)
    # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1|participant) + (1|region) + (1 + zrpe + zpm|unique_elec_id)'
        
    # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1 + zrpe + zpm|participant) + (1 + zrpe + zpm|region) + (1 + zrpe + zpm|unique_elec_id)'
#     formula_zrpe = """
#     tfr ~ 1 + zrpe + zpm + hemi + zrpe * region + zpm * region +
#     (1 + zrpe + zpm | unique_elec_id) +
#     (1 | region)
# """
    #     (1 | participant) +
    # Crossed
#     formula_zrpe = 'tfr ~ 1 + phit + hemi + region + (1 + phit|unique_elec_id) + (1|participant)'
#     categorical = ['hemi', 'region', 'unique_elec_id', 'participant'] 
    
    formula_zrpe = 'tfr ~ 1 + zpm + rpe_resid*phit + hemi + (1|participant) + (1 + zpm + rpe_resid*phit|unique_elec_id)'
    categorical = ['region', 'hemi', 'participant', 'unique_elec_id'] 
    
    label = f'{band}_{analysis_evs}_phit_full_hierarchy_fixed_region'
    model_rpe, results_rpe = run_model(model_df, formula=formula_zrpe, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res=True, label=label, sampler='mcmc')
        
    # for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
    #     res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = results_rpe['posterior']['phit'].mean().values + results_rpe['posterior']['phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

    result_df_summary = az.summary(results_rpe)
    
    phit_values = result_df_summary[result_df_summary.index.str.startswith("rpe_resid:phit|")]['mean'].values
    # pm_values = result_df_summary[result_df_summary.index.str.startswith("zpm|")]['mean'].values

    elec_names = result_df_summary[result_df_summary.index.str.startswith("rpe_resid:phit|")]['mean'].index
    elec_nums = [extract_number(x) for x in elec_names]

    for elec_ix, elec_id in enumerate(elec_nums):
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = result_df_summary[result_df_summary.index=="rpe_resid:phit"]['mean'].values[0] + phit_values[elec_ix]

    res_df['band'] = band 
    model_results.append(res_df)
    
memory_model_df = pd.concat(model_results)
memory_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_zrpe_phit_df_bayes_full_hierarchy_fixed_region', index=False)


In [ ]:
az.plot_trace(results_rpe, var_names=['rpe_resid:phit'])

In [ ]:
analysis_evs = 'feedback_start'
memory_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_phit_df_bayes_full_hierarchy_fixed_region')


In [ ]:
band = 'hfa'

fig, ax = plt.subplots(1, 1, figsize=(4, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_hierarchical.pdf') as pdf:
    sns.barplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.1, color='k')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.015, 0.025])
    pdf.savefig(fig)

In [ ]:
memory_model_df = res_df

In [ ]:
band = 'hfa'

fig, ax = plt.subplots(1, 1, figsize=(4, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_hierarchical.pdf') as pdf:
    sns.barplot(data=memory_model_df[memory_model_df.band==band], x='region', y='beta_phit',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=memory_model_df[memory_model_df.band==band], x='region', y='beta_phit', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.1, color='k')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='memory sensitivity', xlim=[-1,5])
    pdf.savefig(fig)

In [ ]:
gamble_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_rpe_pm_df_bayes_full_hierarchy_fixed_region')



In [ ]:
memory_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_phit_df_bayes_full_hierarchy_fixed_region')



In [ ]:
test_df = gamble_model_df[(gamble_model_df.band=='hfa')]

In [ ]:
test_df['beta_phit'] = memory_model_df[(memory_model_df.band=='hfa')].beta_phit.values

In [ ]:
palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]
palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300, sharex=True, sharey=True)
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_phit_corr.pdf') as pdf:

    sns.regplot(data=test_df[(test_df.region=='dmPFC')], x='beta_rpe_resid', y='beta_phit', color=dmpfc_c, ax=ax[0])
    sns.regplot(data=test_df[(test_df.region=='ACC')], x='beta_rpe_resid', y='beta_phit', color=acc_c, ax=ax[1])
    sns.regplot(data=test_df[(test_df.region=='OFC')], x='beta_rpe_resid', y='beta_phit', color=ofc_c, ax=ax[2])
    sns.regplot(data=test_df[(test_df.region=='HPC')], x='beta_rpe_resid', y='beta_phit', color=hpc_c, ax=ax[3])
    sns.regplot(data=test_df[(test_df.region=='AMY')], x='beta_rpe_resid', y='beta_phit', color=amy_c, ax=ax[4])
    sns.despine()    
    pdf.savefig()
    


In [ ]:
# Are the region-grouped beta values different from 0?
p_values = [] 
spearman_r = []
for region in rois: 
    p_values.append(spearmanr(gamble_model_df[(gamble_model_df.band=='hfa') & (gamble_model_df.region==region)].beta_rpe_resid.values, 
                             gamble_model_df[(gamble_model_df.band=='hfa') & (gamble_model_df.region==region)].beta_phit.values,)[1])
    spearman_r.append(spearmanr(gamble_model_df[(gamble_model_df.band=='hfa') & (gamble_model_df.region==region)].beta_rpe_resid.values, 
                             gamble_model_df[(gamble_model_df.band=='hfa') & (gamble_model_df.region==region)].beta_phit.values,)[0])

In [ ]:
spearman_r

In [ ]:
p_values

In [ ]:
import statsmodels as sm
sm.stats.multitest.fdrcorrection(p_values, alpha=0.05)

In [ ]:
# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
# analysis_evs = 'feedback_start'
# for band in ['hfa']:
#     plot_counter = 0 
#     f, ax = plt.subplots(5,1, dpi=300, figsize=(4,8), sharex=True)
#     label = f'{band}_{analysis_evs}_rpe'
#     with PdfPages(f'{output_dir}/plots/{label}.pdf') as pdf:    
#         for region in ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']:
#             label = f'{region}_{band}_{analysis_evs}_rpe'
            
#             results = az.from_netcdf(f'{output_dir}/{label}_model')
#             ax[plot_counter].set(xlabel='Weight', ylabel='')
#             # f.suptitle(f'{region}_{band}')
#             sns.despine(fig=f, ax=ax[plot_counter], top=True, right=True, left=True, bottom=False)

#             X = ['zrpe']
#             plot_res(results, X, ax=ax[plot_counter])
#             del results
#             ax[plot_counter].set(xlabel = 'model weight', title=f'{band}_{region}')
#             if plot_counter > 0:
#                 ax[plot_counter].set(ylabel=None, yticklabels=[], title=f'{band}_{region}')
#             plot_counter += 1
#         f.tight_layout()
#         pdf.savefig(f)


The t-distribution (robust regression) is significantly, incredibly better than using the gaussian distribution. 

In [ ]:
####### Maximum Likelihood Estimation via statsmodels 

# rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC', 'AINS'] # 'ACC', 'dmPFC'
# bands = ['hfa'] # 'beta', 'alpha'

# model_results_mle = []

# for band in bands: 
#     for region in rois:

#         rel_col = ['tfr', 'zrpe', 'zpm', 'unique_elec_id']
#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

#         res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#         mle_model = MixedLM.from_formula("tfr ~ 1 + zrpe + zpm", 
#                                      groups="unique_elec_id", 
#                                      re_formula="1 + zrpe + zpm", 
#                                      data=model_df).fit()

#         print(region, mle_model.summary())

#         res_df['beta_rpe'] = np.nan
#         for elec_id in res_df.unique_elec_id.unique():
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = mle_model.params['zrpe'] + mle_model.random_effects[elec_id]['zrpe']
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = mle_model.params['zpm'] + mle_model.random_effects[elec_id]['zpm']
            
#         res_df['band'] = band 
        
#         model_results_mle.append(res_df)

In [ ]:
# band = 'hfa'
# analysis_evs = 'feedback_start'

# # let's just load the fixed effects: 

# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

# plot_df = []

# for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):
#     rpe_label = f'{region}_{band}_feedback_start_rpe'
#     mem_label = f'{region}_{band}_encoding_phit_t'

#     rpe_results = az.from_netcdf(f'{output_dir}/{rpe_label}_model')
#     mem_results = az.from_netcdf(f'{output_dir}/{mem_label}_model')

#     results_df = pd.DataFrame(columns=['rpe_sensitivity', 'memory_predictiveness'])

#     results_df['rpe_sensitivity'] = rpe_results.posterior['zrpe'].mean(axis=0)
#     results_df['memory_predictiveness'] = mem_results.posterior['phit'].mean(axis=0)
#     results_df['region'] = region
#     plot_df.append(results_df)
    
# plot_df = pd.concat(plot_df)

# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
# fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

# palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

# with PdfPages(f'{output_dir}/plots/jointplot_rpe_phit.pdf') as pdf:    
#     for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

#     #     sns.jointplot(data=test_df, height=8,
#     #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
#     #                   marginal_kws=dict(fill=True),
#     #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
#         sns.kdeplot(data=plot_df[plot_df.region==region], x="rpe_sensitivity", y="memory_predictiveness", color=palettes[ix], fill=True, 
#                     ax=ax[ix])

#         ax[ix].vlines(0, -0.008, 0.008 ,'k')
#         ax[ix].hlines(0, -0.025, 0.025 ,'k')
#         ax[ix].set(xlim=[-0.025, 0.025], ylim=[-0.008, 0.008], title=f'{region}_feedback_hfa')
#         sns.despine()
        
#     plt.tight_layout()
#     pdf.savefig()
    
#     del rpe_results
#     del mem_results

## Encoding connectivity

In [ ]:
source_region = 'AMY'
metric = 'granger'
band1 = 'hfa'
band2 = 'hfa'
event = 'feedback_start' 
savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'

all_dfs = []
for target_region in ['HPC', 'OFC', 'dmPFC']:

    all_pairs_df = pd.read_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

    all_dfs.append(all_pairs_df)

all_dfs = pd.concat(all_dfs)
# replace RL parameters 
all_dfs.drop(columns=['alpha', 'beta', 'rpe', 'zrpe'], inplace=True) 
new_conn_df = all_dfs.merge(learn_df[['participant', 'trials', 'rpe', 'alpha', 'beta']], on=['participant', 'trials'])



In [ ]:
# for pair_label in new_conn_df[new_conn_df.roi2=='HPC'].pair_label.unique():
#     sns.regplot(data=new_conn_df[(new_conn_df.roi2=='HPC') & (new_conn_df.pair_label==pair_label)], 
#                x='zrpe', 
#                y='conn', 
#                scatter=False, ci=None, color='k', line_kws={'alpha':0.05})

In [ ]:
# TRUE MEM REGRESSION

analysis_evs = 'feedback_start'
rois = ['HPC', 'ACC', 'OFC', 'dmPFC']  # , 'AINS', 'vlPFC', 'dlPFC'
bands = ['theta'] 
model_results = []
df_loos = []
for band in bands: 
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    new_conn_df['zrpe'] = np.nan
    new_conn_df['unique_pair_id'] = np.nan
    # standardize the RPE data
    for subj_id in subj_ids:
        rpe_data = new_conn_df.loc[new_conn_df.participant==subj_id, 'rpe']
        new_conn_df.loc[new_conn_df.participant==subj_id, 'zrpe'] = (rpe_data - np.nanmean(rpe_data)) / (2*np.nanstd(rpe_data))
        # label the unique pairs: 
        for pair_label in new_conn_df[new_conn_df.participant==subj_id].pair_label.unique():
            new_conn_df.loc[(new_conn_df.participant==subj_id) & (new_conn_df.pair_label==pair_label), 
                            'unique_pair_id'] = elec_counter
            elec_counter+=1

    # restrict to electrodes in rois: 
    # new_conn_df = new_conn_df[new_conn_df['roi2'].isin(rois)]
            
    rel_col = ['conn', 'zrpe', 'zpm', 'unique_pair_id', 'participant', 'roi2', 'hemi']
        
    model_df =  new_conn_df[(new_conn_df.band0==band)].reset_index(drop=True)[rel_col].dropna()
        
    res_df = new_conn_df[(new_conn_df.band0==band)].reset_index(drop=True)[['participant', 'pair_label', 'hemi', 'roi2', 'unique_pair_id']].drop_duplicates().copy()

    # RPE second
    # Crossed effects for region across participants, nested effects for electrode across region across participants 
    # Reference: https://stats.stackexchange.com/questions/228800/crossed-vs-nested-random-effects-how-do-they-differ-and-how-are-they-specified
    # Nested (this might be too extreme of an assumption (8/7/2024)
    # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1|participant) + (1|region) + (1 + zrpe + zpm|unique_elec_id)'
        
    # formula_zrpe = 'tfr ~ 1 + zrpe + zpm + (1 + zrpe + zpm|participant) + (1 + zrpe + zpm|region) + (1 + zrpe + zpm|unique_elec_id)'
#     formula_zrpe = """
#     tfr ~ 1 + zrpe + zpm + hemi + zrpe * region + zpm * region +
#     (1 + zrpe + zpm | unique_elec_id) +
#     (1 | region)
# """
    #     (1 | participant) +
    # Crossed
    formula_zrpe = 'conn ~ 1 + zrpe + hemi + roi2 + (1 + zrpe|unique_pair_id) + (1|participant)'
    categorical = ['hemi', 'roi2', 'unique_pair_id', 'participant'] 
    label = f'conn_{analysis_evs}_rpe_pm_full_hierarchy_fixed_region'
    model_rpe, results_rpe = run_model(model_df, formula=formula_zrpe, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=2000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res=True, label=label, sampler='mcmc')
        
    for elec_ix, elec_id in enumerate(res_df.unique_pair_id.unique()):
        res_df.loc[res_df.unique_pair_id==elec_id, 'beta_rpe'] = results_rpe['posterior']['zrpe'].mean().values + results_rpe['posterior']['zrpe|unique_pair_id'].mean(axis=(0,1))[elec_ix].values

    res_df['band'] = band
    model_results.append(res_df)

gamble_model_df = pd.concat(model_results)
# gamble_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_rpe_pm_df_bayes_full_hierarchy_fixed_region', index=False)


In [ ]:
az.plot_trace(results_rpe, var_names=['roi2'])

In [ ]:
gamble_model_df

In [ ]:
band = 'theta'

In [ ]:
g = sns.catplot(data=gamble_model_df[gamble_model_df.band==band], x='hemi', y='beta_rpe', col='roi2', 
            errorbar='se', order = ['l', 'r'], kind='bar', color='k', alpha=0.5)
# g.set(ylim=(-0.01, 0.0175))

f = sns.catplot(data=gamble_model_df[gamble_model_df.band==band], x='hemi', y='beta_rpe', col='roi2', 
            errorbar='se', order = ['l', 'r'], kind='strip', color='k', alpha=0.5)
# f.set(ylim=(-0.01, 0.0175))


## Revised cue analysis

Before I get too into this, I should recognize a weird thing with the z-scoring. Because it's only a subset of the stimuli, if I z-score HERE I will get different value than during feedback. This is bad. Don't mindlessly z-score here.


In [ ]:
analysis_evs = 'cue_on'

In [ ]:
# RPE REGRESSION

rois = ['AMY', 'HPC', 'dmPFC', 'OFC', 'ACC'] 
bands = ['hfa'] 
feature = 'phit'
model_results = []
for band in bands: 
    # RELOAD the data each time the model is run to ensure no funny business with dataframe manipulation between loop runs
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            chan_df = pd.read_csv(f'{filepath}/{analysis_evs}/dfs/{chan}_time_averaged_df.csv')


            chan_df['unique_elec_id'] = elec_counter
            chan_df['urpe'] = np.abs(chan_df.rpe )
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            chan_df['zrpe_unsigned'] = (chan_df.urpe - np.nanmean(chan_df.urpe)) / (2*np.nanstd(chan_df.urpe))
            chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

            elec_counter +=1

            chan_dfs.append(chan_df)

    chan_dfs = pd.concat(chan_dfs)
    chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
    
    # restrict to electrodes in rois: 
    chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]
            
    rel_col = ['tfr', 'zrpe', 'zpm', 'unique_elec_id', 'participant', 'region', 'hemi']
        
    model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
        
    res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'hemi', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    formula_zrpe = 'tfr ~ 1 + zrpe + zpm + hemi + region + (1 + zrpe + zpm|unique_elec_id) + (1|participant)'
    categorical = ['hemi', 'region', 'unique_elec_id', 'participant'] 
    label = f'{band}_{analysis_evs}_rpe_pm_full_hierarchy_fixed_region'
    model_rpe, results_rpe = run_model(model_df, formula=formula_zrpe, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res=True, label=label, sampler='mcmc')
        
    for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = results_rpe['posterior']['zrpe'].mean().values + results_rpe['posterior']['zrpe|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results_rpe['posterior']['zpm'].mean().values + results_rpe['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

    res_df['band'] = band 
    model_results.append(res_df)

cue_model_df = pd.concat(model_results)
cue_model_df.to_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_rpe_pm_df_bayes_full_hierarchy_fixed_region', index=False)


In [ ]:
cue_model_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{analysis_evs}_rpe_pm_df_bayes_full_hierarchy_fixed_region')



In [ ]:
band = 'theta'

fig, ax = plt.subplots(1, 1, figsize=(4, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_rpe_hierarchical.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_rpe',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_rpe', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.1, color='k')

    plt.hlines([0], -1, 5, 'k')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.015, 0.025])
    pdf.savefig(fig)

In [ ]:
# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
# analysis_evs = 'cue_on'
# for band in ['hfa']:
#     plot_counter = 0 
#     f, ax = plt.subplots(5,1, dpi=300, figsize=(4,8), sharex=True)
#     label = f'{band}_{analysis_evs}_phit'
#     with PdfPages(f'{output_dir}/plots/{label}.pdf') as pdf:    
#         for region in ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']:
#             label = f'{region}_{band}_{analysis_evs}_phit_t'
            
#             results = az.from_netcdf(f'{output_dir}/{label}_model')
#             ax[plot_counter].set(xlabel='Weight', ylabel='')
#             # f.suptitle(f'{region}_{band}')
#             sns.despine(fig=f, ax=ax[plot_counter], top=True, right=True, left=True, bottom=False)

#             X = ['phit']
#             plot_res(results, X, ax=ax[plot_counter])
#             del results
#             ax[plot_counter].set(xlabel = 'model weight', title=f'{band}_{region}')
#             if plot_counter > 0:
#                 ax[plot_counter].set(ylabel=None, yticklabels=[], title=f'{band}_{region}')
#             plot_counter += 1
#         f.tight_layout()
#         pdf.savefig(f)


In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'

# let's just load the fixed effects: 

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

plot_df = []

for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):
    rpe_label1 = f'{region}_{band}_feedback_start_rpe'
    rpe_label2 = f'{region}_{band}_cue_on_zrpe_t'

    rpe_results1 = az.from_netcdf(f'{output_dir}/{rpe_label1}_model')
    rpe_results2 = az.from_netcdf(f'{output_dir}/{rpe_label2}_model')

    results_df = pd.DataFrame(columns=['rpe_sensitivity_encoding', 'rpe_sensitivity_cue'])

    results_df['rpe_sensitivity_encoding'] = rpe_results1.posterior['zrpe'].mean(axis=0)
    results_df['rpe_sensitivity_cue'] = rpe_results2.posterior['zrpe'].mean(axis=0)
    results_df['region'] = region
    plot_df.append(results_df)
    
plot_df = pd.concat(plot_df)

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_encoding_rpe_cue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=plot_df[plot_df.region==region], x="rpe_sensitivity_encoding", y="rpe_sensitivity_cue", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.025, 0.025 ,'k')
        ax[ix].hlines(0, -0.025, 0.025 ,'k')
        ax[ix].set(xlim=[-0.025, 0.025], ylim=[-0.025, 0.025], title=f'{region}_feedback_hfa')
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()
    
    del rpe_results1
    del rpe_results2

In [ ]:
rpe_df_feedback = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_feedback_start_rpe_pm_df')
rpe_df_feedback.rename(columns={'beta_rpe': 'rpe_sensitivity_encoding'}, inplace=True)

rpe_df_cue = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_on_zrpe_t_df')
rpe_df_cue.rename(columns={'beta_zrpe': 'rpe_sensitivity_cue'}, inplace=True)

plot_df = pd.merge(rpe_df_feedback, rpe_df_cue)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_rpe_corr.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.regplot(data=plot_df[plot_df.region==region], 
                    x="rpe_sensitivity_encoding", y="rpe_sensitivity_cue", 
                    color=palettes[ix], scatter_kws={'alpha':0.5, 'edgecolor':'black'},
                    ax=ax[ix])
        

#         ax[ix].vlines(0, -0.008, 0.008 ,'k')
#         ax[ix].hlines(0, -0.025, 0.025 ,'k')
        ax[ix].set(title=f'{region}_feedback_hfa')
    # xlim=[-0.08, 0.08], ylim=[-0.015, 0.015], 
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

No electrode-level reactivation of RPE sensitivity. But the information clearly moves around and sticks around in the region.

How do both RPE betas predict cue-related memory activity?

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'

# let's just load the fixed effects: 

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

plot_df = []

for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):
    rpe_label = f'{region}_{band}_feedback_start_rpe'
    mem_label = f'{region}_{band}_cue_on_phit_t'

    rpe_results = az.from_netcdf(f'{output_dir}/{rpe_label}_model')
    mem_results = az.from_netcdf(f'{output_dir}/{mem_label}_model')

    results_df = pd.DataFrame(columns=['rpe_sensitivity_encoding', 'recog_predictiveness_cue'])

    results_df['rpe_sensitivity_encoding'] = rpe_results.posterior['zrpe'].mean(axis=0)
    results_df['recog_predictiveness_cue'] = mem_results.posterior['phit'].mean(axis=0)
    results_df['region'] = region
    plot_df.append(results_df)
    
plot_df = pd.concat(plot_df)

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_encoding_recog_prediction_cue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=plot_df[plot_df.region==region], x="rpe_sensitivity_encoding", y="recog_predictiveness_cue", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.015, 0.015 ,'k')
        ax[ix].hlines(0, -0.025, 0.025 ,'k')
        ax[ix].set(xlim=[-0.025, 0.025], ylim=[-0.015, 0.015], title=f'{region}_feedback_hfa')
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()
    
    del rpe_results
    del mem_results

In [ ]:
band = 'hfa'
analysis_evs = 'feedback_start'

# let's just load the fixed effects: 

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

plot_df = []

for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):
    rpe_label = f'{region}_{band}_cue_on_zrpe_t'
    mem_label = f'{region}_{band}_cue_on_phit_t'

    rpe_results = az.from_netcdf(f'{output_dir}/{rpe_label}_model')
    mem_results = az.from_netcdf(f'{output_dir}/{mem_label}_model')

    results_df = pd.DataFrame(columns=['rpe_sensitivity_cue', 'recog_predictiveness_cue'])

    results_df['rpe_sensitivity_cue'] = rpe_results.posterior['zrpe'].mean(axis=0)
    results_df['recog_predictiveness_cue'] = mem_results.posterior['phit'].mean(axis=0)
    results_df['region'] = region
    plot_df.append(results_df)
    
plot_df = pd.concat(plot_df)

output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_cue_memory_choice_cue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=plot_df[plot_df.region==region], x="rpe_sensitivity_cue", y="recog_predictiveness_cue", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.015, 0.015 ,'k')
        ax[ix].hlines(0, -0.025, 0.025 ,'k')
        ax[ix].set(xlim=[-0.025, 0.025], ylim=[-0.015, 0.015], title=f'{region}_feedback_hfa')
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()
    
    del rpe_results
    del mem_results

In [ ]:
# the following loads the the random_effects + fixed_effects 
band = 'hfa' 
event = 'cue_on'

mem_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_phit_t_df')
rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/{band}_{event}_zrpe_t_df')
rpe_df.rename(columns={'beta_zrpe': 'rpe_sensitivity'}, inplace=True)
mem_df.rename(columns={'beta_phit': 'memory_predictiveness'}, inplace=True)
plot_df = pd.merge(rpe_df, mem_df)


output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_phit_corr_CUE.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.regplot(data=plot_df[plot_df.region==region], 
                    x="rpe_sensitivity", y="memory_predictiveness", 
                    color=palettes[ix], scatter_kws={'alpha':0.5, 'edgecolor':'black'},
                    ax=ax[ix])
        

#         ax[ix].vlines(0, -0.008, 0.008 ,'k')
#         ax[ix].hlines(0, -0.025, 0.025 ,'k')
        ax[ix].set(title=f'{region}_cue_hfa')
    # xlim=[-0.08, 0.08], ylim=[-0.015, 0.015], 
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

In [ ]:
plot_df

In [ ]:
# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
# analysis_evs = 'cue_on'
# for band in ['hfa']:
#     plot_counter = 0 
#     f, ax = plt.subplots(5,1, dpi=300, figsize=(4,8), sharex=True)
#     label = f'{band}_{analysis_evs}_subjective_familiar_t'
#     with PdfPages(f'{output_dir}/plots/{label}.pdf') as pdf:    
#         for region in ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']:
#             label = f'{region}_{band}_{analysis_evs}_subjective_familiar_t'
            
#             results = az.from_netcdf(f'{output_dir}/{label}_model')
#             ax[plot_counter].set(xlabel='Weight', ylabel='')
#             # f.suptitle(f'{region}_{band}')
#             sns.despine(fig=f, ax=ax[plot_counter], top=True, right=True, left=True, bottom=False)

#             X = ['subjective_familiar']
#             plot_res(results, X, ax=ax[plot_counter])
#             del results
#             ax[plot_counter].set(xlabel = 'model weight', title=f'{band}_{region}')
#             if plot_counter > 0:
#                 ax[plot_counter].set(ylabel=None, yticklabels=[], title=f'{band}_{region}')
#             plot_counter += 1
#         f.tight_layout()
#         pdf.savefig(f)


In [ ]:
# band = 'hfa'
# analysis_evs = 'feedback_start'

# # let's just load the fixed effects: 

# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

# plot_df = []

# for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):
#     rpe_label = f'{region}_{band}_feedback_start_rpe'
#     mem_label = f'{region}_{band}_cue_on_subjective_familiar_t'

#     rpe_results = az.from_netcdf(f'{output_dir}/{rpe_label}_model')
#     mem_results = az.from_netcdf(f'{output_dir}/{mem_label}_model')

#     results_df = pd.DataFrame(columns=['rpe_sensitivity_encoding', 'memory_choice_predictiveness_cue'])

#     results_df['rpe_sensitivity_encoding'] = rpe_results.posterior['zrpe'].mean(axis=0)
#     results_df['memory_choice_predictiveness_cue'] = mem_results.posterior['subjective_familiar'].mean(axis=0)
#     results_df['region'] = region
#     plot_df.append(results_df)
    
# plot_df = pd.concat(plot_df)

# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
# fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

# palettes = [dmpfc_c, acc_c, ofc_c, hpc_c, amy_c]

# with PdfPages(f'{output_dir}/plots/jointplot_rpe_encoding_memory_choice_cue.pdf') as pdf:    
#     for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY']):

#     #     sns.jointplot(data=test_df, height=8,
#     #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
#     #                   marginal_kws=dict(fill=True),
#     #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
#         sns.kdeplot(data=plot_df[plot_df.region==region], x="rpe_sensitivity_encoding", y="memory_choice_predictiveness_cue", color=palettes[ix], fill=True, 
#                     ax=ax[ix])

#         ax[ix].vlines(0, -0.01, 0.01 ,'k')
#         ax[ix].hlines(0, -0.025, 0.025 ,'k')
#         ax[ix].set(xlim=[-0.025, 0.025], ylim=[-0.01, 0.01], title=f'{region}_feedback_hfa')
#         sns.despine()
        
#     plt.tight_layout()
#     pdf.savefig()

In [ ]:
# mem_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_subjective_familiar_t_df')
# rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_zrpe_t_df')
# test_df = pd.merge(rpe_df, mem_df)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, amy_c, hpc_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_subjective_familiar_cue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=test_df[test_df.region==region], x="beta_zrpe", y="beta_subjective_familiar", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.035, 0.035 ,'k')
        ax[ix].hlines(0, -0.06, 0.06 ,'k')
        ax[ix].set(xlim=[-0.06, 0.06], ylim=[-0.035, 0.035])
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

So, it seems like amygdala HFA is most important for rpe-related memory. Not necessarily with the same electrodes doing the work, but the region overall increases HFA for both positive RPEs and successful recognition during both encoding and retrieval

# do rpe representations reactivate?

In [ ]:
rpe_df_feedback = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_feedback_start_rpe_pm_df')

rpe_df_cue = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_on_zrpe_t_df')

test_df = pd.merge(rpe_df_feedback, rpe_df_cue)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, amy_c, hpc_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_rpe_encodingvcue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=test_df[test_df.region==region], x="beta_rpe", y="beta_zrpe", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.06, 0.06 ,'k')
        ax[ix].hlines(0, -0.06, 0.06 ,'k')
        ax[ix].set(xlim=[-0.06, 0.06], ylim=[-0.06, 0.06])
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

but still no correlation - the same electrodes do not show high coding at both periods. no reactivation

# do memory representations reactivate?

In [ ]:
phit_df_feedback = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_phit_t_df')

phit_df_cue = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_subjective_familiar_t_df')
# phit_df_cue.rename(columns={'beta_phit':'beta_phit_cue'}, inplace=True)

test_df = pd.merge(phit_df_feedback, phit_df_cue)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, amy_c, hpc_c]

with PdfPages(f'{output_dir}/plots/jointplot_phit_subjective_familiar_encodingvcue.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=test_df[test_df.region==region], x="beta_phit", y="beta_subjective_familiar", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.035, 0.035 ,'k')
        ax[ix].hlines(0, -0.035, 0.035 ,'k')
        ax[ix].set(xlim=[-0.035, 0.035], ylim=[-0.035, 0.035])
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

In [ ]:
sns.regplot(data=test_df[test_df.region=='dmPFC'], x='beta_phit', y='beta_subjective_familiar')

no reactivation of phit either

## does beta rpe at encoding predict phit at retrieval? 

In [ ]:
rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_feedback_start_rpe_pm_df')

phit_df_cue = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_subjective_familiar_t_df')
# phit_df_cue.rename(columns={'beta_phit':'beta_phit_cue'}, inplace=True)

test_df = pd.merge(rpe_df, phit_df_cue)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
fig, ax = plt.subplots(1, 5, figsize=(21, 4), dpi=300)

palettes = [dmpfc_c, acc_c, ofc_c, amy_c, hpc_c]

with PdfPages(f'{output_dir}/plots/jointplot_rpe_subjective_familiar_encodig.pdf') as pdf:    
    for ix, region in enumerate(['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']):

    #     sns.jointplot(data=test_df, height=8,
    #                   x="beta_rpe", y="beta_phit", hue="region", kind='kde', joint_kws={'alpha':1}, 
    #                   marginal_kws=dict(fill=True),
    #                   hue_order=['HPC', 'AMY', 'OFC', 'ACC', 'dmPFC'], palette=[hpc_c, amy_c, ofc_c, acc_c, dmpfc_c])
        sns.kdeplot(data=test_df[test_df.region==region], x="beta_rpe", y="beta_subjective_familiar", color=palettes[ix], fill=True, 
                    ax=ax[ix])

        ax[ix].vlines(0, -0.035, 0.035 ,'k')
        ax[ix].hlines(0, -0.065, 0.065 ,'k')
        ax[ix].set(xlim=[-0.065, 0.065], ylim=[-0.035, 0.035])
        sns.despine()
        
    plt.tight_layout()
    pdf.savefig()

In [ ]:
sns.regplot(data=test_df[test_df.region=='ACC'], x='beta_rpe', y='beta_subjective_familiar')


If anything, electrodes that code RPE more during encoding are LESS likely to increase during successful recognition

## does beta phit at encoding predict beta_rpe at cue

In [ ]:
rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_on_zrpe_t_df')

phit_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_phit_t_df')

test_df = pd.merge(rpe_df, phit_df)


In [ ]:
sns.regplot(data=test_df[test_df.region=='ACC'], x='beta_phit', y='beta_zrpe')


no

## what about the relationship between encoding beta_rpe and false alarm HFA?

In [ ]:
# fa_ha_df = chan_dfs[(chan_dfs.fband=='hfa') &(chan_dfs.memory_outcome=='false_alarms')].groupby(['unique_elec_id']).mean().reset_index()
# fa_ha_df.rename(columns={'tfr': 'fa_average_hfa'}, inplace=True)

preject_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_preject_t_df')

rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_feedback_start_rpe_pm_df')

test_df = pd.merge(preject_df[['unique_elec_id', 'beta_preject']], rpe_df, on=['unique_elec_id'])

test_df.dropna(subset='beta_preject', inplace=True)


In [ ]:
model_df = test_df[test_df.region=='ACC']
model = smf.mixedlm(formula = 'beta_preject ~ 1 + beta_rpe', data=model_df, groups=model_df['participant']).fit()
model.summary()


no relationship between ENCODING rpe encoding and CUE CR/FA

## what is the relationship between cue phit and FA hfa? 



In [ ]:
fa_ha_df = chan_dfs[(chan_dfs.fband=='hfa') &(chan_dfs.memory_outcome=='correct_rejections')].groupby(['unique_elec_id']).mean().reset_index()
fa_ha_df.rename(columns={'tfr': 'fa_average_hfa'}, inplace=True)

# preject_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/hfa_cue_on_preject_t_df')


rpe_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_on_zrpe_t_df')

test_df = pd.merge(fa_ha_df, rpe_df, on=['unique_elec_id'])
# test_df = pd.merge(preject_df[['unique_elec_id', 'beta_preject']], rpe_df, on=['unique_elec_id'])

# test_df.dropna(subset='beta_preject', inplace=True)



In [ ]:
model_df = test_df[test_df.region=='OFC']
model = smf.mixedlm(formula = 'beta_preject ~ 1 + beta_zrpe', data=model_df, groups=model_df['participant']).fit()
model.summary()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=300)

sns.regplot(data=test_df[test_df.region.isin(['AMY'])], x='beta_zrpe', y='fa_average_hfa', 
            ax=ax, color=amy_c)

sns.despine()


only in the amygdala do we find evidence that electrodes increasing HFA to RPE information at CUE

have less HFA during false alarms 



# Generally speaking: HFA during false alarms warrants more investigation!!

In [ ]:
b

## Detect RPE encoding after cue presentation during recall

In [ ]:
analysis_evs = ['cue_on']
chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
    
    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')
                
            
            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            if event in ['cue_on', 'recog_time']:
                chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                # Make a new response variable, memory outcome 
                chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')                
#                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
#                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
            else:
                chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))

                
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            
            elec_counter +=1
            
            chan_dfs.append(chan_df)
        
chan_dfs = pd.concat(chan_dfs)
chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
# chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
# chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)
chan_dfs['subjective_familiar'] = chan_dfs.apply(lambda x: 1 if (x.memory_outcome=='hits') | (x.memory_outcome=='false_alarms') else -1, axis=1)
chan_dfs['objective_memory'] = chan_dfs.apply(lambda x: 1 if (x.memory_outcome=='hits') | (x.memory_outcome=='correct_rejections') else -1, axis=1)


In [ ]:
from scipy.stats import ttest_ind

In [ ]:
test_df = chan_dfs[(chan_dfs.region=='OFC') & (chan_dfs.fband=='beta')]
ts_subj = []
for unique_elec in test_df.unique_elec_id.unique():
    t, p = ttest_ind(test_df[(test_df.unique_elec_id==unique_elec) & (test_df.memory_outcome=='hits')].tfr,
          test_df[(test_df.unique_elec_id==unique_elec) & (test_df.memory_outcome=='false_alarms')].tfr)
    ts_subj.append(t)

In [ ]:
sns.regplot(x=gamble_model_df[(gamble_model_df.region=='OFC') & (gamble_model_df.band=='beta')].beta_rpe, 
           y=ts_subj)

In [ ]:
spearmanr(gamble_model_df[(gamble_model_df.region=='OFC') & (gamble_model_df.band=='beta')].beta_rpe,
         ts_subj)

In [ ]:
test_df = chan_dfs[(chan_dfs.region=='AMY') & (chan_dfs.fband=='hfa')]
ts_subj = []
for unique_elec in test_df.unique_elec_id.unique():
    t, p = ttest_ind(test_df[(test_df.unique_elec_id==unique_elec) & (test_df.subjective_familiar==1)].tfr,
          test_df[(test_df.unique_elec_id==unique_elec) & (test_df.subjective_familiar==-1)].tfr)
    ts_subj.append(t)
    

In [ ]:
test_df = chan_dfs[(chan_dfs.region=='AMY') & (chan_dfs.fband=='hfa')]
ts_obj = []
for unique_elec in test_df.unique_elec_id.unique():
    t, p = ttest_ind(test_df[(test_df.unique_elec_id==unique_elec) & (test_df.objective_memory==1)].tfr,
          test_df[(test_df.unique_elec_id==unique_elec) & (test_df.objective_memory==-1)].tfr)
    ts_obj.append(t)
    

In [ ]:
ttest_1samp(ts_subj, popmean=0, nan_policy='omit')


In [ ]:
test_df = chan_dfs[(chan_dfs.region=='HPC') & (chan_dfs.fband=='hfa')].groupby(['unique_elec_id', 'subjective_familiar']).mean().reset_index() 
from scipy.stats import ttest_rel, ttest_ind, ttest_1samp
ttest_rel(test_df[test_df.subjective_familiar==1].tfr, 
         test_df[test_df.subjective_familiar==-1].tfr)



In [ ]:
sns.catplot(data=chan_dfs[(chan_dfs.region=='AMY') & (chan_dfs.fband=='hfa')].reset_index(drop=True), x='objective_memory', y='tfr', kind='bar', errorbar='se')


In [ ]:
sns.catplot(data=chan_dfs[(chan_dfs.region=='HPC') & (chan_dfs.fband=='hfa')].reset_index(drop=True), x='subjective_familiar', y='tfr', kind='bar', errorbar='se')


In [ ]:
chan_dfs[chan_dfs.region=='AMY'].reset_index(drop=True)

In [ ]:
chan_dfs['memory_outcome'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome in ['hits', 'correct_rejections'] else -1 if x.memory_outcome in ['misses', 'false_alarms'] else np.nan, axis=1)
# 

In [ ]:
sns.lmplot(data=chan_dfs[(chan_dfs.region=='AMY') & (chan_dfs.fband=='hfa')], x='zrpe', y='tfr', scatter=False)


In [ ]:
chan_dfs[(chan_dfs.region=='AMY') & (chan_dfs.fband=='hfa') & (chan_dfs.rpe>0)].tfr.mean()

In [ ]:
# RPE REGRESSION

rois = ['AMY', 'HPC', 'dmPFC', 'OFC', 'ACC'] 
bands = ['beta', 'hfa'] 
model_results = []
for band in bands:         
    # LOAD the data
    analysis_evs = ['cue_on']
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for event in analysis_evs:
            for chan in elec_df.label.unique():
                # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                chan_df['unique_elec_id'] = elec_counter
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                if event in ['cue_on', 'recog_time']:
                    chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                    # Make a new response variable, memory outcome 
                    chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                           'false_alarms']].idxmax(1).to_frame('memory_outcome')                
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                else:
                    chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                elec_counter +=1

                chan_dfs.append(chan_df)

    chan_dfs = pd.concat(chan_dfs)
    chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
    # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
    # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
    chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
    chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)

    
    # restrict to electrodes in rois: 
    chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]

    rel_col = ['tfr', 'zrpe', 'zpm', 'phit', 'unique_elec_id', 'hemi', 'region']
    model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
    res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    formula = 'tfr ~ 1 + zrpe + zpm + phit + zrpe:phit + hemi + region + (1 + zrpe + zpm + phit + zrpe:phit|unique_elec_id)'
    categorical = ['hemi', 'region'] 
    label = f'{band}_cue_rpe_pm_phit_only_t_df'
    model, results = run_model(model_df, formula=formula, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res = True, label=label)

    for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_zrpe_phit'] = results['posterior']['zrpe:phit'].mean().values + results['posterior']['zrpe:phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
#         res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

    res_df['band'] = band 
    model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_rpe_pm_phit_only_t_df', index=False)



In [ ]:
test_df = model_df[model_df.unique_elec_id==0]

In [ ]:
formula = 'tfr ~ 1 + zrpe +zpm'
categorical = None
label = f'{band}_test'
model_test, results_test = run_model(test_df, formula=formula, categorical=categorical,
                           priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                           output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                           return_model=True, save_model_res = False, label=label)



In [ ]:
cue_model_df

In [ ]:
az.plot_trace(results_test)
plt.tight_layout()

In [ ]:
# RPE REGRESSION

rois = ['AMY', 'HPC', 'dmPFC', 'OFC', 'ACC'] 
bands = ['beta', 'hfa'] 
model_results = []
for band in bands:         
    # LOAD the data
    analysis_evs = ['cue_on']
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for event in analysis_evs:
            for chan in elec_df.label.unique():
                # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                chan_df['unique_elec_id'] = elec_counter
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                if event in ['cue_on', 'recog_time']:
                    chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                    # Make a new response variable, memory outcome 
                    chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                           'false_alarms']].idxmax(1).to_frame('memory_outcome')                
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                else:
                    chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                elec_counter +=1

                chan_dfs.append(chan_df)

    chan_dfs = pd.concat(chan_dfs)
    chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
    # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
    # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
    chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
    chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)

    
    # restrict to electrodes in rois: 
    chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]

    rel_col = ['tfr', 'zrpe', 'zpm', 'unique_elec_id', 'hemi', 'region']
    model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
    res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    formula = 'tfr ~ 1 + zrpe + zpm + hemi + region + (1 + zrpe + zpm|unique_elec_id)'
    categorical = ['hemi', 'region'] 
    label = f'{band}_encoding_rpe_pm_only_t'
    model, results = run_model(model_df, formula=formula, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res = True, label=label)

    for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = results['posterior']['zrpe'].mean().values + results['posterior']['zrpe|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

    res_df['band'] = band 
    model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_rpe_pm_only_t_df', index=False)



In [ ]:
# RPE REGRESSION

rois = ['AMY', 'HPC', 'dmPFC', 'OFC', 'ACC'] 
bands = ['beta', 'hfa'] 
model_results = []
for band in bands:         
    # LOAD the data
    analysis_evs = ['cue_on']
    chan_dfs = [] 
    glms = []
    elec_counter = 0 
    for subj_id in subj_ids:
        print(f'Loading subj {subj_id}')
        day = 'Day1'
        filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
        elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')

        for event in analysis_evs:
            for chan in elec_df.label.unique():
                # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                chan_df['unique_elec_id'] = elec_counter
                chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                if event in ['cue_on', 'recog_time']:
                    chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                    # Make a new response variable, memory outcome 
                    chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                           'false_alarms']].idxmax(1).to_frame('memory_outcome')                
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
    #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                else:
                    chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                elec_counter +=1

                chan_dfs.append(chan_df)

    chan_dfs = pd.concat(chan_dfs)
    chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
    # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
    # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
    chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
    chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)

    
    # restrict to electrodes in rois: 
    chan_dfs = chan_dfs[chan_dfs['region'].isin(rois)]

    rel_col = ['tfr', 'zrpe', 'zpm', 'unique_elec_id', 'hemi', 'region']
    model_df =  chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[rel_col].dropna()
    res_df = chan_dfs[(chan_dfs.fband==band)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

    formula = 'tfr ~ 1 + phit + hemi + region + (1 + phit |unique_elec_id)'
    categorical = ['hemi', 'region'] 
    label = f'{band}_encoding_phit_only_t'
    model, results = run_model(model_df, formula=formula, categorical=categorical,
                               priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                               return_model=True, save_model_res = True, label=label)

    for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
        res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = results['posterior']['phit'].mean().values + results['posterior']['phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values

    res_df['band'] = band 
    model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/cue_phit_only_t_df', index=False)



In [ ]:
# ####### Bayesian regression via bambi 

# rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
# bands = ['hfa', 'beta', 'alpha', 'theta']  
# model_results = []
# for band in bands: 
#     for region in rois:
        
#         analysis_evs = ['cue_on']
#         chan_dfs = [] 
#         glms = []
#         elec_counter = 0 
#         for subj_id in subj_ids:
#             print(f'Loading subj {subj_id}')
#             day = 'Day1'
#             filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
#             elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


#             for event in analysis_evs:
#                 for chan in elec_df.label.unique():
#                     # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
#                     # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
#                     chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


#                     chan_df['unique_elec_id'] = elec_counter
#                     chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
#                     if event in ['cue_on', 'recog_time']:
#                         chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
#                         # Make a new response variable, memory outcome 
#                         chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
#                                                'false_alarms']].idxmax(1).to_frame('memory_outcome')                
#         #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
#         #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
#                     else:
#                         chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


#                     chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#                     chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
#                     chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

#                     elec_counter +=1

#                     chan_dfs.append(chan_df)

#         chan_dfs = pd.concat(chan_dfs)
#         chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
#         # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
#         # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
#         chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
#         chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)


#         # modeling portion 
        
#         rel_col = ['tfr', 'zpm', 'zrpe', 'unique_elec_id', 'hemi']
#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
#         res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#         # restrict to right hemisphere:
        
#         formula = 'tfr ~ 1 + zrpe + zpm  + hemi + (1 + zrpe + zpm|unique_elec_id)'
#         categorical = ['hemi']
#         label = f'{region}_{band}_zrpe_zpm_t_cue'
#         model, results = run_model(model_df, formula=formula, categorical=categorical,
#                                    priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
#                                    output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
#                                    return_model=True, save_model_res = True, label=label)

#         res_df['beta_zrpe'] = np.nan
#         for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_rpe'] = results['posterior']['zrpe'].mean().values + results['posterior']['zrpe|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
#         res_df['band'] = band 
#         model_results.append(res_df)
        
# cue_model_df = pd.concat(model_results)
# cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_zpm_t_cue_df', index=False)


In [ ]:
band = 'beta'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_zrpe_phit.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_zrpe_phit',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_zrpe_phit', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')
#     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
#     sns.boxplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False)
    plt.hlines([0], -1, 5, 'r')
    ax.set(ylabel='memory sensitivity', xlim=[-1,5], ylim=[-0.06, 0.06])
    pdf.savefig(fig)

In [ ]:
ttest_1samp(cue_model_df[(cue_model_df.region=='AMY') & (cue_model_df.band=='beta')].beta_phit.values, popmean=0)


In [ ]:
# band = 'beta'
# fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
# sns.despine()
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_rpe.pdf') as pdf:
#     sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_rpe',errorbar='se', 
#             order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
#     sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_rpe', 
#         order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')
# #     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
# #             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
# #     sns.boxplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
# #         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False)
#     plt.hlines([0], -1, 5, 'r')
#     ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.06, 0.06])
#     pdf.savefig(fig)

In [ ]:
band = 'beta'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_pm.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_pm',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_pm', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')
#     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
#     sns.boxplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False)
    plt.hlines([0], -1, 5, 'r')
    ax.set(ylabel='PM sensitivity', xlim=[-1,5], ylim=[-0.015, 0.015])
    pdf.savefig(fig)

In [ ]:
cue_model_full_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_phit_t_cue_df')

In [ ]:
band = 'hfa'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/cue_{band}_rpe_phit.pdf') as pdf:
    sns.barplot(data=cue_model_full_df[cue_model_full_df.band==band], x='region', y='beta_zrpe_memory',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_full_df[cue_model_full_df.band==band], x='region', y='beta_zrpe_memory', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')
#     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
#     sns.boxplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False)
    plt.hlines([0], -1, 5, 'r')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.07, 0.04])
    pdf.savefig(fig)

In [ ]:
merge_df1 = pd.merge(gamble_model_df, cue_model_full_df, on=['participant', 'region', 'channel', 'unique_elec_id', 'band'], suffixes=['_feedback', '_cue'])


In [ ]:
merge_df1

In [ ]:
sns.regplot(x=merge_df1[(merge_df1.region=='ACC')].beta_rpe, 
           y=merge_df1[(merge_df1.region=='ACC')].beta_zrpe_memory)



In [ ]:
sns.regplot(data=merged_df[(merged_df.region=='AMY') & (merged_df.band=='hfa')],
           x='beta_rpe_feedback', y='beta_rpe_cue')

In [ ]:
merged_df

So Amygdala HFA elevates for RPE in both encoding and cue. But it does so really homogenously. There's no evidence that amygdala HFA activation during encoding correlates with amygdala HFA activation during cue (e.g. a less responsive elec during encoding is less responsive during cue). 

What does this mean? Amygdala as an on-off salience detector but not material to memory?


Look for reactivation of RPE coding across conditions (irrespective of memory). 

In [ ]:
gamble_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_rpe_pm_only_t_df')

cue_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_zpm_t_cue_df')



In [ ]:
cue_model_df.band.unique()

In [ ]:
merged_df = pd.merge(gamble_model_df, cue_model_df, on=['participant', 'region', 'channel', 'unique_elec_id', 'band'], suffixes=['_feedback', '_cue'])


In [ ]:
sns.lmplot(data=merged_df, x='beta_rpe_feedback', y='beta_rpe_cue', hue='region', col='band', scatter=False)
# plt.vlines(0, -0.015, 0.015, 'k')
# plt.hlines(0, -0.06, 0.06, 'k')

There is no reactivation effect. What I mean is that we don't see the same pattern for encoding RPE as cue RPE in any frequency band or region (within-frequency)

In [ ]:
from scipy.stats import pearsonr, spearmanr

In [ ]:
region = 'ACC'

In [ ]:
spearmanr(merged_df[(merged_df.region==region) & (merged_df.band=='hfa')].beta_rpe_feedback, 
        merged_df[(merged_df.region==region) & (merged_df.band=='alpha')].beta_rpe_cue)


1) dmPFC theta RPE encoding predicts dmPFC beta RPE cue (but both are negative)

2) HPC alpha RPE encoding predicts HPC beta RPE cue (encoding is negative but cue is positive)

3) OFC theta RPE encoding negatively predicts OFC beta RPE cue 

In summary, no positive encoding comes back as positive encoding. There is no reactivation even across bands

# NO EVIDENCE OF RPE-RELATED REACTIVATION AFTER CUE PRESENTATION

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['hfa', 'beta', 'alpha', 'theta']  
model_results = []
for band in bands: 
    for region in rois:
        
        analysis_evs = ['cue_on']
        chan_dfs = [] 
        glms = []
        elec_counter = 0 
        for subj_id in subj_ids:
            print(f'Loading subj {subj_id}')
            day = 'Day1'
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


            for event in analysis_evs:
                for chan in elec_df.label.unique():
                    # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                    # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                    chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                    chan_df['unique_elec_id'] = elec_counter
                    chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                    if event in ['cue_on', 'recog_time']:
                        chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                        # Make a new response variable, memory outcome 
                        chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                               'false_alarms']].idxmax(1).to_frame('memory_outcome')                
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                    else:
                        chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                    chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                    elec_counter +=1

                    chan_dfs.append(chan_df)

        chan_dfs = pd.concat(chan_dfs)
        chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
        # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
        # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
        chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
        chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)


        # modeling portion 
        
        rel_col = ['tfr', 'phit', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        # restrict to right hemisphere:
        
        formula = 'tfr ~ 1 + phit  + hemi + (1 + phit|unique_elec_id)'
        categorical = ['hemi']
        label = f'{region}_{band}_phit_t_cue'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_phit'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = results['posterior']['phit'].mean().values + results['posterior']['phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/phit_t_cue_df', index=False)


In [ ]:
cue_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_zpm_t_cue_df')

In [ ]:
cue_model_df

In [ ]:
band = 'hfa'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_phit_cue.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_phit',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_phit', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')

    plt.hlines([0], -1, 5, 'r')
    ax.set(ylabel='memory sensitivity', xlim=[-1,5], ylim=[-0.05, 0.05])
    pdf.savefig(fig)

In [ ]:
band = 'hfa'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/encoding_{band}_pm_cue.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_pm',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_pm', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')

    plt.hlines([0], -1, 5, 'r')
    ax.set(ylabel='PM sensitivity', xlim=[-1,5], ylim=[-0.07, 0.04])
    pdf.savefig(fig)

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['hfa']  
model_results = []
for band in bands: 
    for region in rois:
        
        analysis_evs = ['cue_on']
        chan_dfs = [] 
        glms = []
        elec_counter = 0 
        for subj_id in subj_ids:
            print(f'Loading subj {subj_id}')
            day = 'Day1'
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


            for event in analysis_evs:
                for chan in elec_df.label.unique():
                    # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                    # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                    chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                    chan_df['unique_elec_id'] = elec_counter
                    chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                    if event in ['cue_on', 'recog_time']:
                        chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                        # Make a new response variable, memory outcome 
                        chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                               'false_alarms']].idxmax(1).to_frame('memory_outcome')                
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                    else:
                        chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                    chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                    elec_counter +=1

                    chan_dfs.append(chan_df)

        chan_dfs = pd.concat(chan_dfs)
        chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
        # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
        # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
        chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
        chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)


        # modeling portion 
        
        rel_col = ['tfr', 'zrpe', 'phit', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        # restrict to right hemisphere:
        
        formula = 'tfr ~ 1 + zrpe + hemi + phit + zrpe:phit + (1 + zrpe + phit + zrpe:phit|unique_elec_id)'
        categorical = ['hemi']
        label = f'{region}_{band}_zrpe_phit_t_cue'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_zrpe_memory'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_zrpe_memory'] = results['posterior']['zrpe:phit'].mean().values + results['posterior']['zrpe:phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_pm'] = results['posterior']['zpm'].mean().values + results['posterior']['zpm|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_phit_t_cue_df', index=False)


# Detect p(hit) sensitivity during cue period

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['hfa', 'beta']  
model_results = []
for band in bands: 
    for region in rois:
        
        analysis_evs = ['cue_on']
        chan_dfs = [] 
        glms = []
        elec_counter = 0 
        for subj_id in subj_ids:
            print(f'Loading subj {subj_id}')
            day = 'Day1'
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


            for event in analysis_evs:
                for chan in elec_df.label.unique():
                    # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                    # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                    chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                    chan_df['unique_elec_id'] = elec_counter
                    chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                    if event in ['cue_on', 'recog_time']:
                        chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                        # Make a new response variable, memory outcome 
                        chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                               'false_alarms']].idxmax(1).to_frame('memory_outcome')                
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                    else:
                        chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                    chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                    elec_counter +=1

                    chan_dfs.append(chan_df)

        chan_dfs = pd.concat(chan_dfs)
        chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
        # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
        # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
        chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
        chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)


        # modeling portion 
        
        rel_col = ['tfr', 'phit', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        # restrict to right hemisphere:
        
        formula = 'tfr ~ 1 + phit + hemi + (1 + phit|unique_elec_id)'
        categorical = ['hemi']
        label = f'{region}_{band}_phit_t_cue'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_phit'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = results['posterior']['phit'].mean().values + results['posterior']['phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
cue_model_df = pd.concat(model_results)
cue_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/phit_t_cue_df', index=False)


# Detect p(reject) sensitivity during cue period 

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['hfa', 'beta']  
model_results = []
for band in bands: 
    for region in rois:
        
        analysis_evs = ['cue_on']
        chan_dfs = [] 
        glms = []
        elec_counter = 0 
        for subj_id in subj_ids:
            print(f'Loading subj {subj_id}')
            day = 'Day1'
            filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')


            for event in analysis_evs:
                for chan in elec_df.label.unique():
                    # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
                    # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
                    chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')


                    chan_df['unique_elec_id'] = elec_counter
                    chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
                    if event in ['cue_on', 'recog_time']:
                        chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                        # Make a new response variable, memory outcome 
                        chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                               'false_alarms']].idxmax(1).to_frame('memory_outcome')                
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
        #                 chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
                    else:
                        chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))


                    chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                    chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
                    chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]

                    elec_counter +=1

                    chan_dfs.append(chan_df)

        chan_dfs = pd.concat(chan_dfs)
        chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
        # chan_dfs['hit_binom'] = chan_dfs['hits'].astype(int)
        # chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5
        chan_dfs['phit'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='hits' else -1 if x.memory_outcome=='misses' else np.nan, axis=1)
        chan_dfs['preject'] = chan_dfs.apply(lambda x: 1 if x.memory_outcome=='correct_rejections' else -1 if x.memory_outcome=='false_alarms' else np.nan, axis=1)


        # modeling portion 
        
        rel_col = ['tfr', 'preject', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        # restrict to right hemisphere:
        
        formula = 'tfr ~ 1 + preject + hemi + (1 + preject|unique_elec_id)'
        categorical = ['hemi']
        label = f'{region}_{band}_preject_t_cue'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_preject'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_preject'] = results['posterior']['preject'].mean().values + results['posterior']['preject|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
cue_model_df2 = pd.concat(model_results)
cue_model_df2.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/preject_t_cue_df', index=False)


In [ ]:
cue_model_df1 = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/phit_t_cue_df')
cue_model_df2 = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/preject_t_cue_df')
cue_model_df3 = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_phit_t_cue_df')
# cue_model_df3 = cue_model_df3[cue_model_df3.band=='hfa']


In [ ]:
cue_model_df1

Is there a relationship between the electrodes that are predictive of successful recognition vs. those predictive of successful rejection? 

In [ ]:
# Let's plot the correlation between phit sens and preject sensitivity 

In [ ]:
plot_df = pd.merge(cue_model_df1, cue_model_df2)

# .merge(cue_model_df3)

In [ ]:
gamble_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/encoding_rpe_phit_intx_t_df')

In [ ]:
sns.barplot(data=gamble_model_df[gamble_model_df.band=='hfa'], x='region', y='beta_rpe_phit',errorbar='se', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
sns.stripplot(data=gamble_model_df[gamble_model_df.band=='hfa'], x='region', y='beta_rpe_phit', 
    order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')


In [ ]:
sns.barplot(data=plot_df[plot_df.band=='hfa'], x='region', y='beta_phit',errorbar='se', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
sns.stripplot(data=plot_df[plot_df.band=='hfa'], x='region', y='beta_phit', 
    order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')


In [ ]:
sns.barplot(data=cue_model_df3[cue_model_df3.band=='hfa'], x='region', y='beta_zrpe_memory',errorbar='se', 
        order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha = 0.5, color='k')
sns.stripplot(data=cue_model_df3[cue_model_df3.band=='hfa'], x='region', y='beta_zrpe_memory', 
    order = ['dmPFC', 'ACC', 'OFC', 'HPC', 'AMY'], alpha=0.5, color='k')


In [ ]:
sns.jointplot(data=plot_df[plot_df['region'].isin(['AMY', 'HPC']) & (plot_df.band=='hfa')], 
              x="beta_phit", y="beta_preject", hue="region", joint_kws={'alpha':0.3})
plt.vlines(0, -0.015, 0.01 ,'k')
plt.hlines(0, -0.025, 0.045 ,'k')

In [ ]:
sns.jointplot(data=plot_df[plot_df['region'].isin(['AMY', 'HPC']) & (plot_df.band=='beta')], 
              x="beta_phit", y="beta_preject", hue="region", joint_kws={'alpha':0.3})
plt.vlines(0, -0.02, 0.04 ,'k')
plt.hlines(0, -0.03, 0.01 ,'k')

In [ ]:
plot_df

In [ ]:
sns.jointplot(data=plot_df[~plot_df['region'].isin(['AMY', 'HPC']) & (plot_df.band=='hfa')], 
              x="beta_phit", y="beta_preject", hue="region", joint_kws={'alpha':0.3})
plt.vlines(0, -0.015, 0.01 ,'k')
plt.hlines(0, -0.025, 0.045 ,'k')

In [ ]:
sns.jointplot(data=plot_df[~plot_df['region'].isin(['AMY', 'HPC']) & (plot_df.band=='beta')], 
              x="beta_phit", y="beta_preject", hue="region", joint_kws={'alpha':0.3})
plt.vlines(0, -0.02, 0.04 ,'k')
plt.hlines(0, -0.03, 0.01 ,'k')

In [ ]:

band = 'hfa'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/recog_{band}_zrpe_memory_cue.pdf') as pdf:
    sns.barplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_zrpe_memory',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], alpha = 0.5, color='k')
    sns.stripplot(data=cue_model_df[cue_model_df.band==band], x='region', y='beta_zrpe_memory', 
        order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], alpha=0.5, color='k')
#     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
#     sns.boxplot(data=recog_model_df[recog_model_df.band==band], x='region', y='beta_zrpe_memory', 
#         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False, ax=ax)
    plt.hlines([0], -1, 6, 'r')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.07, 0.04])
    pdf.savefig(fig)

In [ ]:
# ####### Bayesian regression via bambi 

# rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
# bands = ['theta', 'alpha', 'beta', 'hfa']  # 
# model_results = []
# for band in bands: 
#     for region in rois:
        
#         rel_col = ['tfr', 'phit', 'unique_elec_id', 'hemi']
#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
#         res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#         formula = 'tfr ~ 1 + phit + hemi + (1 + phit|unique_elec_id)'
#         categorical = None 
#         label = f'{region}_{band}_phit_gaussian'
#         model, results = run_model(model_df, formula=formula, categorical=categorical,
#                                    priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='gaussian', 
#                                    output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
#                                    return_model=True, save_model_res = True, label=label)

#         res_df['beta_phit'] = np.nan
#         for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_phit'] = results['posterior']['phit'].mean().values + results['posterior']['phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
#         res_df['band'] = band 
#         model_results.append(res_df)
        
# phit_model_df = pd.concat(model_results)
# phit_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/phit_gaussian_channel_df', index=False)


In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['theta', 'alpha', 'beta', 'hfa']  
model_results = []
for band in bands: 
    for region in rois:
        
        rel_col = ['tfr', 'memory_outcome', 'zrpe', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        formula = 'tfr ~ 1 + zrpe + memory_outcome + zrpe:memory_outcome + hemi + (1 + zrpe + memory_outcome + zrpe:memory_outcome|unique_elec_id)'
        categorical = None 
        label = f'{region}_{band}_zrpe_memory_outcome_t'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_zrpe_memory'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_zrpe_memory'] = results['posterior']['zrpe:memory_outcome'].mean().values + results['posterior']['zrpe:memory_outcome|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
recog_model_df = pd.concat(model_results)
recog_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_memory_outcome_t_channel_df', index=False)


In [ ]:
# ####### Bayesian regression via bambi 

# rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
# bands = ['hfa']  
# model_results = []
# for band in bands: 
#     for region in rois:
        
#         rel_col = ['tfr', 'phit', 'zrpe', 'unique_elec_id', 'hemi']
#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
#         res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

#         formula = 'tfr ~ 1 + zrpe + phit + zrpe:phit + hemi + (1 + zrpe + phit + zrpe:phit|unique_elec_id)'
#         categorical = None 
#         label = f'{region}_{band}_zrpe_phit_gaussian'
#         model, results = run_model(model_df, formula=formula, categorical=categorical,
#                                    priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
#                                    output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
#                                    return_model=True, save_model_res = True, label=label)

#         res_df['beta_zrpe_memory'] = np.nan
#         for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
#             res_df.loc[res_df.unique_elec_id==elec_id, 'beta_zrpe_memory'] = results['posterior']['zrpe:phit'].mean().values + results['posterior']['zrpe:phit|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
#         res_df['band'] = band 
#         model_results.append(res_df)
        
# recog_model_df = pd.concat(model_results)
# recog_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_phit_gaussian_channel_df', index=False)


In [ ]:
# for band in bands[-1:]: 
#     for region in rois:
#         phit_label = f'{region}_{band}_zrpe_phit_gaussian'
#         phit_results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/{phit_label}_model') 
#         memory_outcome_label = f'{region}_{band}_zrpe_memory_outcome_gaussian'
#         memory_outcome_results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/{memory_outcome_label}_model') 

#         model_dict = {'phit':phit_results, 
#                       'memory_outcome':memory_outcome_results}
        
#         df_loo = az.compare(model_dict)
#         az.plot_compare(df_loo)
        
#         del phit_results
#         del memory_outcome_results
#         del model_dict

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['theta', 'alpha', 'beta', 'hfa']  
model_results = []
for band in bands: 
    for region in rois:
        
        rel_col = ['tfr', 'memory_outcome', 'zpm', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        formula = 'tfr ~ 1 + zpm + memory_outcome + zpm:memory_outcome + hemi + (1 + zpm + memory_outcome + zpm:memory_outcome|unique_elec_id)'
        categorical = None 
        label = f'{region}_{band}_zrpe_memory_gaussian'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='t', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_zpm_memory'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_zpm_memory'] = results['posterior']['zpm:memory_outcome'].mean().values + results['posterior']['zpm:memory_outcome|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
recog_model_df = pd.concat(model_results)
recog_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zpm_memory_outcome_t_channel_df', index=False)


In [ ]:
recog_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_memory_gaussian_channel_df')

In [ ]:
fband = 'hfa' 
region = 'HPC'
model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
sns.lmplot(data=model_df, x='zrpe', y='tfr', hue='phit', scatter=False, ci=95)

In [ ]:
fband = 'hfa' 
region = 'dmPFC'
model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
sns.lmplot(data=model_df, x='zrpe', y='tfr', hue='phit', scatter=False, ci=95)

In [ ]:
fband = 'hfa' 
region = 'AMY'
model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
sns.lmplot(data=model_df, x='zrpe', y='tfr', hue='phit', scatter=False, ci=95)

In [ ]:

band = 'hfa'
fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=300)
sns.despine()
with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/recog_{band}_zrpe_memory.pdf') as pdf:
    sns.barplot(data=recog_model_df[recog_model_df.band==band], x='region', y='beta_zrpe_memory',errorbar='se', 
            order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], alpha = 0.5, color='k')
    sns.stripplot(data=recog_model_df[recog_model_df.band==band], x='region', y='beta_zrpe_memory', 
        order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], alpha=0.5, color='k')
#     sns.violinplot(data=gamble_model_df[gamble_model_df.band==band], x='region', y='beta_rpe', 
#             order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'])
#     sns.boxplot(data=recog_model_df[recog_model_df.band==band], x='region', y='beta_zrpe_memory', 
#         order = ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC'], showfliers=False, ax=ax)
    plt.hlines([0], -1, 6, 'r')
    ax.set(ylabel='RPE sensitivity', xlim=[-1,5], ylim=[-0.07, 0.04])
    pdf.savefig(fig)

5/18/2024: I don't really need to add electrode-by-electrode level analysis here because all the HPC electrodes are RPE sensitive. So there's not a lot of variation to look at for more- vs. less-RPE sensitive electrodes. 

In [ ]:
recog_model_df

In [ ]:
####### Bayesian regression via bambi 

rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC'] 
bands = ['theta', 'alpha', 'beta', 'hfa'] 
model_results = []
for band in bands: 
    for region in rois:
        
        rel_col = ['tfr', 'preject', 'unique_elec_id', 'hemi']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
        res_df = chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[['participant', 'region', 'channel', 'unique_elec_id']].drop_duplicates().copy()

        formula = 'tfr ~ 1 + preject + hemi + (1 + preject|unique_elec_id)'
        categorical = None 
        label = f'{region}_{band}_reject_only_gaussian'
        model, results = run_model(model_df, formula=formula, categorical=categorical,
                                   priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='gaussian', 
                                   output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                                   return_model=True, save_model_res = True, label=label)

        res_df['beta_memory'] = np.nan
        for elec_ix, elec_id in enumerate(res_df.unique_elec_id.unique()):
            res_df.loc[res_df.unique_elec_id==elec_id, 'beta_memory'] = results['posterior']['preject'].mean().values + results['posterior']['preject|unique_elec_id'].mean(axis=(0,1))[elec_ix].values
            
        res_df['band'] = band 
        model_results.append(res_df)
        
reject_model_df = pd.concat(model_results)
reject_model_df.to_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/reject_only_gaussian_channel_df', index=False)


In [ ]:
results['posterior']['memory_outcome'].mean().values

In [ ]:
results['posterior']['memory_outcome'].mean().values

In [ ]:
recog_model_df = pd.read_csv('/sc/arion/work/qasims01/MemoryBanditData/EMU/zrpe_memory_gaussian_channel_df')


In [ ]:
results_df = gamble_model_df.merge(recog_model_df, on=['participant','region', 'channel', 'unique_elec_id', 'band'])



In [ ]:
band = 'hfa'

for roi in ['HPC', 'OFC', 'dmPFC', 'ACC', 'AMY']:
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{roi}_{band}_recog_rpe.pdf') as pdf:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=300)
        sns.despine()

        data = results_df[(results_df.region==roi) & (results_df.band==band)]

        rpe_model = smf.mixedlm(formula='beta_zrpe_memory ~ 1 + beta_rpe', 
                           data=data,
                           re_formula='beta_rpe',
                           groups=data['participant']).fit()

        sns.regplot(data = data, x= 'beta_rpe', y='beta_zrpe_memory', 
                    scatter=True, ci=95, ax=ax)

    #     r, p = spearmanr(data['beta_rpe'], data['beta_memory'])

    #         if roi in ['AINS']:
    #             ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
    #                    title = f'{roi} {band}',
    #                   xlim = [-0.1, 0.15], ylim = [-0.15, 0.05])
    #         else:
        ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
           title = f'{roi} {band}')

#         model, b_dict, se_dict, z_dict, p_dict, CI_dict = bootstrap_mixedlm(formula='beta_zrpe_memory ~ 1 + beta_rpe', 
#                              data=data, 
#                              re_formula='beta_rpe', 
#                              group_key='participant',
#                              n_boot=250)

#         ax.text(.01, .98, 'group_beta={:.2f}, p={:.2g}'.format(b_dict['beta_rpe'], 
#                                                      p_dict['beta_rpe']),
#                 transform=ax.transAxes)

        plt.tight_layout()
        pdf.savefig(fig)
#         plt.close()
        

In [ ]:
band = 'beta'
for roi in ['dmPFC', 'ACC', 'OFC', 'AMY', 'HPC']:
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{roi}_{band}_recog_rpe.pdf') as pdf:

        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=300)
        sns.despine()

        data = results_df[(results_df.region==roi) & (results_df.band==band)]

        rpe_model = smf.mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                           data=data,
                           re_formula='beta_rpe',
                           groups=data['participant']).fit()

        sns.regplot(data = data, x= 'beta_rpe', y='beta_memory',
                    scatter=True, ci=95, ax=ax)

        if roi in ['AINS']:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
                   title = f'{roi} {band}')
        else:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
               title = f'{roi} {band}')

        model, b_dict, se_dict, z_dict, p_dict, CI_dict = bootstrap_mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                             data=data, 
                             re_formula='beta_rpe', 
                             group_key='participant',
                             n_boot=250)

        ax.text(.01, .98, 'group_beta={:.2f}, p={:.2g}'.format(b_dict['beta_rpe'], 
                                                     p_dict['beta_rpe']),
                transform=ax.transAxes)

        plt.tight_layout()
        pdf.savefig(fig)
#         plt.close()

In [ ]:
band = 'theta'
for roi in ['HPC', 'dmPFC', 'OFC', 'ACC', 'AMY']:
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{roi}_{band}_recog_rpe.pdf') as pdf:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=300)
        sns.despine()

        data = results_df[(results_df.region==roi) & (results_df.band==band)]

        rpe_model = smf.mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                           data=data,
                           re_formula='beta_rpe',
                           groups=data['participant']).fit()

        sns.regplot(data = data, x= 'beta_rpe', y='beta_memory', 
                    scatter=True, ci=95, ax=ax)

        if roi in ['AINS']:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
                   title = f'{roi} {band}')
        else:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
               title = f'{roi} {band}')

        model, b_dict, se_dict, z_dict, p_dict, CI_dict = bootstrap_mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                             data=data, 
                             re_formula='beta_rpe', 
                             group_key='participant',
                             n_boot=250)

        ax.text(.01, .98, 'group_beta={:.2f}, p={:.2g}'.format(b_dict['beta_rpe'], 
                                                     p_dict['beta_rpe']),
                transform=ax.transAxes)

        plt.tight_layout()
        pdf.savefig(fig)
#         plt.close()
    

In [ ]:
band = 'alpha'
for roi in ['HPC', 'OFC', 'ACC', 'dmPFC', 'AMY']:
    with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/{roi}_{band}_recog_rpe.pdf') as pdf:

        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=300)
        sns.despine()

        data = results_df[(results_df.region==roi) & (results_df.band==band)]

#         if roi in ['ACC']:
#             data = data[data.participant!='MS020']

        rpe_model = smf.mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                           data=data,
                           re_formula='beta_rpe',
                           groups=data['participant']).fit()

        sns.regplot(data = data, x= 'beta_rpe', y='beta_memory', 
                    scatter=True, ci=95, ax=ax)

        if roi in ['AINS']:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
                   title = f'{roi} {band}')
        else:
            ax.set(ylabel='memory sensitivity', xlabel='RPE sensitivity', 
               title = f'{roi} {band}')

        model, b_dict, se_dict, z_dict, p_dict, CI_dict = bootstrap_mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                             data=data, 
                             re_formula='beta_rpe', 
                             group_key='participant',
                             n_boot=250)

        ax.text(.01, .98, 'group_beta={:.2f}, p={:.2g}'.format(b_dict['beta_rpe'], 
                                                     p_dict['beta_rpe']),
                transform=ax.transAxes)

        plt.tight_layout()
        pdf.savefig(fig)
#         plt.close()

In [ ]:
# Shawn's idea: bootstrap the model with replacement 
from IPython.display import clear_output
from tqdm import tqdm
import scipy as sp

def bootstrap_mixedlm(formula, 
                     data, 
                     re_formula, 
                     group_key,
                     n_boot=500):
    """
    Wrapper for statsmodels mixedlm function that computed bootstrap corrected CI and p-values for mixedlm
    
    """
    
    def _resample_group(data, key='participant'):
        group = data[key].unique()

        # Resample group with replacement
        resampled_group = np.random.choice(group, size=len(group), replace=True)

        resampled_df = [] 
        for ix, group in enumerate(resampled_group): 
            # filter down to group selected for sample 
            temp_df = data[data[key]==group]
            # provide a unique ID for the resampled group
            temp_df[key] = ix
            # append the group to df
            resampled_df.append(temp_df)

        resampled_df = pd.concat(resampled_df)

        return resampled_df
    
    model = smf.mixedlm(formula=formula, 
                       data=data,
                       re_formula=re_formula,
                       groups=data[group_key]).fit()
    
    boot_res = {f'{x}': [] for x in model.params.keys()}
    
    for i in tqdm(range(n_boot)):

        resampled_df = _resample_group(data, key=group_key)

        boot_model = smf.mixedlm(formula=formula, 
                           data=resampled_df,
                           re_formula=re_formula,
                           groups=resampled_df[group_key]).fit()

        for key in boot_model.params.keys():
            boot_res[key].append(boot_model.params[key])
        
        clear_output(wait=True)
        
    b_dict = {}
    se_dict = {}
    z_dict = {}
    p_dict = {}
    CI_dict = {}
    for key in model.params.keys():
        new_beta = np.mean(boot_res[key])
        b_dict[key] = new_beta
        new_se = np.std(boot_res[key])
        se_dict[key] = new_se
        new_z = new_beta/new_se
        z_dict[key] = new_z 
#         # two-tailed test on z
        new_p = sp.stats.norm.sf(abs(new_z))*2
        p_dict[key] = new_p 
        CI_dict[key] = np.percentile(boot_res[key], [2.5, 97.5])
        
    
    return model, b_dict, se_dict, z_dict, p_dict, CI_dict

In [ ]:
model, b_dict, se_dict, z_dict, p_dict, CI_dict = bootstrap_mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
                     data=data, 
                     re_formula='beta_rpe', 
                     group_key='participant',
                     n_boot=1000)

In [ ]:
model.summary()

In [ ]:
b_dict

In [ ]:
p_dict

In [ ]:
z_dict

In [ ]:
# # cross-validate the models: does dropping one subject change the results a lot? If so, use the model without that subject. 

# betas = [] 
# for participant in data.participant.unique():
#     df = data[data.participant!=participant].copy()

# #     # Shuffle 'y' within each group
# #     df['beta_memory'] = df.groupby('participant')['beta_memory'].transform(lambda x: np.random.permutation(x))

# #     # Shuffle 'x' within each group
# #     df['beta_rpe'] = df.groupby('participant')['beta_rpe'].transform(lambda x: np.random.permutation(x))

#     # Shuffle groups
# #     group_list = df['participant'].unique()
# #     np.random.shuffle(group_list)
# #     df['participant'] = df['participant'].map(dict(zip(df['participant'].unique(), group_list)))
    
#     rpe_model = smf.mixedlm(formula='beta_memory ~ 1 + beta_rpe', 
#                        data=df,
#                        groups=df['participant']).fit()
    
#     betas.append(rpe_model.params['beta_rpe']/rpe_model.bse['beta_rpe'])
    
# np.where(np.abs(zscore(betas)) > 3)[0][0]

## ENCODING POWER:

In [ ]:
region = 'HPC'
band = 'theta' 

rel_col = ['tfr', 'zrpe', 'zpm', 'phit', 'unique_elec_id', 'participant']

model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

# logistic regression:

#         formula = 'hit_binom ~ 1 + zrpe + zpm + ztfr + zAge + sex + zrpe:ztfr + zpm:ztfr + (1 + zrpe + zpm|participant) + (1 + ztfr + zrpe:ztfr + zpm:ztfr|unique_elec_id)'

# linear regression
# 
formula = 'tfr ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|unique_elec_id)'

# +zrpe+zpm+phit+zrpe:phit+zpm:phit

categorical = ['phit']
# 'sex', 'hemi', 

label = f'{region}_{band}_encoding_fully_hierarchical_t'
# _encoding_rand_elec_int_only_t

#         test_mod = smf.mixedlm(formula='tfr ~ 1 + zrpe + zpm + C(phit) + zrpe:C(phit) + zpm:C(phit)', 
#                        data=model_df, 
#                        groups=model_df['unique_elec_id']).fit()

#         model_results_mle[band][region].append(test_mod)


# model, results = run_model(model_df, formula=formula, categorical=categorical,
#       priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.9, model_fam='t', 
#       output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)

In [ ]:
    # construct the model 
    model = bmb.Model(formula=formula, 
              data=model_df,
             family='t',
             priors=None,
             categorical=categorical)
    
#     # fit the model 
#     results=model.fit(cores=cores, 
#                           chains=chains, 
#                           tune=tune, 
#                           draws=draws, 
#                           target_accept=target_accept,
#                          inference_method='nuts_numpyro',
#                          idata_kwargs={"log_likelihood": True})

In [ ]:
model.build()

In [ ]:
model.graph()

In [ ]:
results2=model.fit(cores=4, 
                      chains=4, 
                      tune=1000, 
                      draws=1000, 
                      target_accept=0.8,
                     inference_method='nuts_numpyro',
                     idata_kwargs={"log_likelihood": True})

In [ ]:
model_dict = {'full_hierarchical_slopes_only': results,
             'electrode_hierarchical_slopes_only': results1,
             'electrode_hierarhical_both': results2}

In [ ]:
df_loo = az.compare(model_dict)

In [ ]:
az.plot_compare(df_loo)

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results1, X, ax=ax)

In [ ]:
model_df = chan_dfs[(chan_dfs.fband=='hfa') & (chan_dfs.region=='HPC')].reset_index(drop=True)[rel_col].dropna()


In [ ]:
np.nanmin(model_df.tfr)

In [ ]:
plt.hist(model_df.tfr)

In [ ]:
rois = ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC', 'AINS'] # 'ACC', 'dmPFC'
bands = ['theta', 'alpha', 'beta', 'hfa'] # 'beta', 'alpha'

model_results_mle = {f'{a}': {f'{b}': [] for b in rois} for a in bands}

for band in bands: # , 'theta', 'alpha', 'beta'
    for region in rois: # 'OFC', 'dmPFC', 'HPC', 'AMY', 'ACC'

        rel_col = ['tfr', 'zrpe', 'zpm', 'zAge', 'hemi', 'sex', 'unique_elec_id']
        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

        # logistic regression:

#         formula = 'hit_binom ~ 1 + zrpe + zpm + ztfr + zAge + sex + zrpe:ztfr + zpm:ztfr + (1 + zrpe + zpm|participant) + (1 + ztfr + zrpe:ztfr + zpm:ztfr|unique_elec_id)'

        # linear regression
        # 
        formula = 'tfr ~ 1 + zrpe + zpm + zAge + sex + (1 + zrpe + zpm|unique_elec_id)'
        
        # +zrpe+zpm+phit+zrpe:phit+zpm:phit
        # zpm + phit + zrpe:phit + zpm:phit +
    
        categorical = ['sex', 'hemi']
        # 'sex', 'hemi', 
        
        label = f'{region}_{band}_encoding_rpe_pm_only_gaussian'
        # _encoding_rand_elec_int_only_t
        
#         test_mod = smf.mixedlm(formula='tfr ~ 1 + zrpe + zpm + C(phit) + zrpe:C(phit) + zpm:C(phit)', 
#                        data=model_df, 
#                        groups=model_df['unique_elec_id']).fit()
        
#         model_results_mle[band][region].append(test_mod)


        model, results = run_model(model_df, formula=formula, categorical=categorical,
              priors=None, cores=4, chains=4, tune=3000, draws=2000, target_accept=0.9, model_fam='gaussian', 
              output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)
#         del results

In [ ]:
test_df = chan_dfs[(chan_dfs.fband=='hfa') & (chan_dfs.region=='OFC')].reset_index(drop=True)[rel_col].dropna()


In [ ]:
plt.hist(test_df.tfr)

In [ ]:
test = smf.mixedlm(formula='tfr ~ 1 + zrpe + zpm + C(phit) + zrpe:C(phit) + zpm:C(phit)', 
                       data=model_df, 
                       groups=model_df['unique_elec_id'])

In [ ]:
test.fit?

In [ ]:
smf.mixedlm?

In [ ]:
test_mod.summary()

In [ ]:
for band in ['hfa']:
    for region in ['HPC']: 
        f, ax = plt.subplots(1,2, dpi=300, figsize=(8,4), sharex=True, sharey=True)
        
        rel_col = ['ztfr', 'zrpe', 'zpm', 'hit_binom', 'zAge', 'hemi', 'sex', 'unique_elec_id', 'participant']

        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()
    
        model_df['zrpe_cat'] = pd.qcut(model_df['zrpe'], q=2)
        model_df['zpm_cat'] = pd.qcut(model_df['zpm'], q=2)
        
        sns.regplot(data=model_df[model_df.hit_binom==0], x='zrpe', y='ztfr', scatter=False, ci=68, ax=ax[0])
        sns.regplot(data=model_df[model_df.hit_binom==1], x='zrpe', y='ztfr', scatter=False, ci=68, ax=ax[0])
        sns.regplot(data=model_df[model_df.hit_binom==0], x='zpm', y='ztfr', scatter=False, ci=68, ax=ax[1])
        sns.regplot(data=model_df[model_df.hit_binom==1], x='zpm', y='ztfr', scatter=False, ci=68, ax=ax[1])
        sns.despine()
        

In [ ]:
dm = model.response_component.design

In [ ]:
np.where(np.array(dm.response)>0.5)

In [ ]:
# Design matrix for fixed effects: 
sns.heatmap(np.array(dm.common)[:, 1:])

In [ ]:
# Compute the variance inflation factor to show that the fixed-effects  are not multicollinear
[variance_inflation_factor(np.array(dm.common), i) 
               for i in range(np.array(dm.common).shape[1])]

In [ ]:
# Design matrix for random effects: 
sns.heatmap(np.array(dm.group))

If I want to plot posterior predictive comparisons:

In [ ]:
# Set model inputs 
region = 'HPC'
band = 'hfa'
formula = 'tfr ~ 1 + zrpe + zpm + phit + zAge + sex + hemi + zrpe:phit + zpm:phit + (1|unique_elec_id)' 
rel_col = ['tfr', 'zrpe', 'zpm', 'phit', 'zAge', 'hemi', 'sex', 'unique_elec_id', 'participant']
categorical=['sex', 'phit']
model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

# (re)make model
model = bmb.Model(formula=formula, 
          data=model_df,
         family='t',
         priors=None,
         categorical=categorical) 

# Load idata 
label = f'{region}_{band}_encoding_rand_elec_int_only_t'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/{label}_model') 

In [ ]:
with PdfPages(f'{save_dir_plots}/encoding_regression_pps_RPE_{region}_{band}.pdf') as pdf:    
    bmb.interpret.plot_predictions(
        model, 
        results, 
        conditional=["zrpe"],
        sample_new_groups = True,
        use_hdi=True,
        pps=False,
        subplot_kwargs={"main": "zrpe"},
        legend=False,
        fig_kwargs={"figsize": (4, 4), 'sharey':True}
    )
    sns.despine()
    pdf.savefig()

In [ ]:
with PdfPages(f'{save_dir_plots}/encoding_regression_pps_PM_{region}_{band}.pdf') as pdf:    
    bmb.interpret.plot_predictions(
        model, 
        results, 
        conditional=["zpm", "phit"],
        sample_new_groups = True,
        use_hdi=True,
        pps=False,
        subplot_kwargs={"main": "zpm", "group": "phit", "panel": "phit"},
        legend=False,
        fig_kwargs={"figsize": (4, 4), 'sharey':True}
    )
    sns.despine()
    pdf.savefig()

Do a quick model comparison between the random effects for subjects vs. the random effects for electrodes. The latter removes a lot of results...

In [ ]:
fully_hierarchical_gauss = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/OFC_hfa_encoding_fully_hierarchical_gaussian_model') 
fully_hierarchical_t = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/OFC_hfa_encoding_fully_hierarchical_t_model') 

# t_dist_int_only_subj = az.from_netcdf('/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels/OFC_hfa_encoding_rand_subj_int_only_t_model') 

model_comp = {'fully_hierarchical_gauss': fully_hierarchical_gauss,
              'fully_hierarchical_t': fully_hierarchical_t}

df_comp = az.compare(model_comp)

az.plot_compare(df_comp)


In [ ]:
del fully_hierarchical_t
del fully_hierarchical_gauss
del df_comp

In [ ]:
az.plot_trace(t_dist)
plt.tight_layout()

1/18/24: I fixed the electrodes for the Iowa patients (determining out of brain and wm electrodes from the manual assignments) and removed noisy events.

In [ ]:
save_dir_plots

1/31/24: I have changed my baselining method. I've also fixed a couple of syncs that were off by a bit. 

In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

for band in ['theta', 'alpha', 'beta', 'hfa']:
    plot_counter = 0 
    f, ax = plt.subplots(1,6, dpi=300, figsize=(16,4), sharex=True)
    with PdfPages(f'{save_dir_plots}/encoding_regression_posteriors_{band}_encoding_rpe_pm_only_gaussian.pdf') as pdf:    
        for region in ['HPC', 'OFC', 'AMY', 'ACC', 'dmPFC', 'AINS']:
            label = f'{region}_{band}_encoding_rpe_pm_only_gaussian_model'
            results = az.from_netcdf(f'{output_dir}/{label}')
            ax[plot_counter].set(xlabel='Weight', ylabel='')
            # f.suptitle(f'{region}_{band}')
            sns.despine(fig=f, ax=ax[plot_counter], top=True, right=True, left=True, bottom=False)

            X = ['zrpe',
                'zpm']
            plot_res(results, X, ax=ax[plot_counter])
            del results
            ax[plot_counter].set(xlabel = 'model weight', title=f'{band}_{region}')
            if plot_counter > 0:
                ax[plot_counter].set(ylabel=None, yticklabels=[], title=f'{band}_{region}')
            plot_counter += 1
        f.tight_layout()
        pdf.savefig(f)


## What do we find with the electrode-level slopes?

In [ ]:
region = 'OFC'
band = 'theta'
label = f'{region}_{band}_encoding_rpe_pm_only_gaussian_model'
results = az.from_netcdf(f'{output_dir}/{label}')
summary = az.summary(results, hdi_prob=0.95, var_names=["zrpe|unique_elec_id"], filter_vars="like")


4/1/2024: final thoughts

1. HFA effects in hpc (RPE) and now also ofc (RPE:memory) and amygdala (RPE:memory). The inclusion of amygdala here is perfectly convergent with Valerie Voon's work: https://www.jneurosci.org/content/42/13/2756

2. Theta effects in ofc (RPE:memory) and now also in dmPFC (PM:memory). How do I interpret the latter?

3. Alpha effects all over for PM: memory. This seems interesting and worth investigating separately... 

Can I now find the strongest electrodes?

In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

for band in ['theta']:
    with PdfPages(f'{save_dir_plots}/encoding_regression_posteriors_{band}_encoding_fully_hierarchical_t.pdf') as pdf:    
        for region in ['OFC']:
            label = f'{region}_{band}_encoding_fully_hierarchical_t_model'
            results = az.from_netcdf(f'{output_dir}/{label}')
            summary = az.summary(results, hdi_prob=0.95, var_names=["zrpe|unique_elec_id"], filter_vars="like")
            del results

In [ ]:
summary

In [ ]:
import re
# Sort the electrodes from LOW to HIGH individual beta values 
sorted_elecs = [int(re.search(r"\[(\d+)\]", x).group(1)) if re.search(r"\[(\d+)\]", x) else None for x in summary.sort_values(by=['mean']).index]


In [ ]:
sorted_elecs

In [ ]:
chan_dfs[(chan_dfs.unique_elec_id==1159)].participant

In [ ]:
chan_dfs[(chan_dfs.unique_elec_id==1159)].channel

In [ ]:
# Generate a custom color palette with 60 colors

custom_palette = sns.color_palette("Purples_r", n_colors=len(sorted_elecs))

for ix, elec_id in enumerate(sorted_elecs):
    sns.regplot(data=chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='theta')],
            x='zrpe', y='tfr', scatter=False, ci=None, color = 'k', line_kws={'alpha': 0.1})
    

In [ ]:
sns.regplot(data=chan_dfs[(chan_dfs.region=='OFC') & (chan_dfs.fband=='theta')],
        x='zrpe', y='tfr', scatter=False, ci=95, color = 'k', line_kws={'alpha': 0.1})


In [ ]:
sns.regplot(data=chan_dfs[(chan_dfs.unique_elec_id==725) & (chan_dfs.fband=='hfa')],
            x='zrpe', y='tfr', scatter=False, ci=95, color=custom_palette[ix], line_kws={'alpha': 0.5})
    

Now do the same thing for an interaction effect

In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'

for band in ['hfa']:
    with PdfPages(f'{save_dir_plots}/encoding_regression_posteriors_{band}_encoding_fully_hierarchical_t.pdf') as pdf:    
        for region in ['AMY']:
            label = f'{region}_{band}_encoding_fully_hierarchical_t_model'
            results = az.from_netcdf(f'{output_dir}/{label}')
            summary = az.summary(results, hdi_prob=0.95, var_names=["zrpe:phit|unique_elec_id"], filter_vars="like")
            del results

In [ ]:
summary

In [ ]:
import re
# Sort the electrodes from LOW to HIGH individual beta values 
sorted_elecs = [int(re.search(r", (\d+)\]", x).group(1)) if re.search(r", (\d+)\]", x) else None for x in summary.sort_values(by=['mean']).index]


In [ ]:
# Generate a custom color palette with 60 colors

custom_palette = sns.color_palette("Purples", n_colors=61)

for ix, elec_id in enumerate(sorted_elecs):
    sns.lmplot(data=chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='hfa')],
            x='zrpe', y='tfr', hue='phit', scatter=False, ci=None, line_kws={'alpha': 0.5})
    

In [ ]:
elec_id = 186

In [ ]:
sns.lmplot(data=chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='hfa')],
            x='zrpe', y='tfr', hue='phit', scatter=False, ci=95, line_kws={'alpha': 0.5})

In [ ]:
elec_id = 188

In [ ]:
sns.lmplot(data=chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='hfa')],
            x='zrpe', y='tfr', hue='phit', scatter=False, ci=95, line_kws={'alpha': 0.5})

In [ ]:
chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='hfa')]

In [ ]:
chan_dfs[(chan_dfs.unique_elec_id==elec_id) & (chan_dfs.fband=='hfa')]

## ENCODING AMP CONNECTIVITY:

In [ ]:
source_region = 'dmPFC'
target_region = 'OFC'
metric = 'amp'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}/{source_region}_{target_region}_{metric}_hfa_hfa_feedback_start_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
sns.barplot(data=all_pairs_df, x='rpe', y='conn',  hue = 'errorbar='se')

synchronized HPC-OFC HFA amplitudes are bad for salience memory 

In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
# [(all_pairs_df.good_epoch==1)]
model_df = all_pairs_df[rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_hfa_hfa_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, model_fam='gaussian', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
test_df = az.summary(results)

In [ ]:
test_df.head(10)

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

amplitude connectivity diminishes when encoding stimuli successfully and diminishes further for more salient stimuli


4/19/2024: Can I make sure that 'IED' epochs don't show different patterns of connectivity then those without? 

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', hue = 'good_epoch', scatter=False)

In [ ]:
all_pairs_df.good_epoch.sum()/all_pairs_df.shape[0]

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', hue = 'good_epoch', scatter=False)

In [ ]:
all_pairs_df.good_epoch.sum()/all_pairs_df.shape[0]

In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', hue = 'good_epoch', scatter=False)

In [ ]:
all_pairs_df.good_epoch.sum()/all_pairs_df.shape[0]

Let's limit this to pairs where at least one electrode has significant theta power during encoding (2 SDs above 1/f baseline) 

In [ ]:
theta_channel_ids = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/theta_channel_df')

all_pairs_df['theta_peak'] = 0 
for row in theta_channel_ids.itertuples():
    all_pairs_df.loc[(all_pairs_df.participant==row.participant) & (all_pairs_df.pair_label.str.contains(row.channel)), 'theta_peak'] = 1
    
    

In [ ]:
sns.regplot(data=all_pairs_df.groupby('participant').mean().reset_index(), x='good_epoch', y='dprime', scatter=True)


In [ ]:
all_pairs_df.groupby('participant').mean().reset_index()

In [ ]:
all_pairs_df.keys()

In [ ]:
sns.lmplot(data=all_pairs_df, x='zrpe', y='conn', row = 'good_epoch', col='theta_peak', hue='phit', scatter=False)

In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[(all_pairs_df.theta_peak==1)][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t_model'


f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
with PdfPages(f'{save_dir_plots}/encoding_regression_posteriors_{source_region}_{target_region}_{metric}_encoding_fully_hierarchical_t.pdf') as pdf:    
    results = az.from_netcdf(f'{output_dir}/{label}')
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)

    X = ['zrpe',
         'zpm',
         'phit', 
         'zrpe:phit',
         'zpm:phit']

    plot_res(results, X, ax=ax)
    del results
    ax.set(xlabel = 'model weight', title=f'{source_region}_{target_region}_{metric}')
    f.tight_layout()
    pdf.savefig(f)

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5

In [ ]:
theta_channel_ids = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/theta_channel_df')

all_pairs_df['theta_peak'] = 0 
for row in theta_channel_ids.itertuples():
    all_pairs_df.loc[(all_pairs_df.participant==row.participant) & (all_pairs_df.pair_label.str.contains(row.channel)), 'theta_peak'] = 1
    
    

In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[(all_pairs_df.good_epoch==1)  & (all_pairs_df.theta_peak==1)][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t_model'


f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
with PdfPages(f'{save_dir_plots}/encoding_regression_posteriors_{source_region}_{target_region}_{metric}_encoding_fully_hierarchical_t.pdf') as pdf:    
    results = az.from_netcdf(f'{output_dir}/{label}')
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)

    X = ['zrpe',
         'zpm',
         'phit', 
         'zrpe:phit',
         'zpm:phit']

    plot_res(results, X, ax=ax)
    del results
    ax.set(xlabel = 'model weight', title=f'{source_region}_{target_region}_{metric}')
    f.tight_layout()
    pdf.savefig(f)

And a nice, significant effect of HPC-OFC theta coherence for RPE!

And now, HPC-AMY theta coherence? 

In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
theta_channel_ids = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/theta_channel_df')

all_pairs_df['theta_peak'] = 0 
for row in theta_channel_ids.itertuples():
    all_pairs_df.loc[(all_pairs_df.participant==row.participant) & (all_pairs_df.pair_label.str.contains(row.channel)), 'theta_peak'] = 1
    
    

In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[(all_pairs_df.good_epoch==1)  & (all_pairs_df.theta_peak==1)][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

## Encoding Granger Causality

Just one-directional granger causality

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'gc'
band1 = 'theta'
band2 = 'theta'
event = 'feedback'

all_pairs_df = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/granger/HPC_OFC_granger_theta_theta_feedback_start_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'participant']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|participant)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'granger'
band1 = 'theta'
band2 = 'theta'
event = 'feedback'

all_pairs_df = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/granger/AMY_OFC_granger_theta_theta_feedback_start_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'participant']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|participant)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_encoding_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
sns.lmplot(data=model_df, x='zpm', y='conn', hue='phit', scatter=False, ci=95)


In [ ]:
sns.lmplot(data=model_df, x='zrpe', y='conn', hue='phit', scatter=False, ci=95)


## RETRIEVAL:

In [ ]:
analysis_evs = ['recog_time']

chan_dfs = [] 
glms = []
elec_counter = 0 
for subj_id in subj_ids:
    print(f'Loading subj {subj_id}')
    day = 'Day1'
    filepath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/scratch/TFR'
    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/{day}_reref_elec_df')
    
    
    for event in analysis_evs:
        for chan in elec_df.label.unique():
            # for chan in tqdm(ch_names, ascii=True, desc='Loading TFR dataframes'): 
            # chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df_SME.csv')
            chan_df = pd.read_csv(f'{filepath}/{event}/dfs/{chan}_time_averaged_df.csv')
            chan_df['unique_elec_id'] = elec_counter
            chan_df['zrpe'] = (chan_df.rpe - np.nanmean(chan_df.rpe)) / (2*np.nanstd(chan_df.rpe))
            if event == 'recog_time':
                chan_df['zpm'] = (chan_df.DPRIME_y - np.nanmean(chan_df.DPRIME_y)) / (2*np.nanstd(chan_df.DPRIME_y))
                # Make a new response variable, memory outcome 
                chan_df['memory_outcome'] = chan_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome')                
                chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['misses', 'false_alarms'])] = 0
                chan_df['memory_outcome'][chan_df['memory_outcome'].isin(['hits', 'correct_rejections'])] = 1
            else:
                chan_df['zpm'] = (chan_df.DPRIME - np.nanmean(chan_df.DPRIME)) / (2*np.nanstd(chan_df.DPRIME))

            # try:
            #     chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt'])
            # except KeyError:
            #     chan_df['gamble_rt_log'] = np.log(chan_df['gamble_rt_gamble'])
            chan_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            chan_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]
            chan_df['hemi'] = elec_df[elec_df.label==chan].hemisphere.str.lower().values[0]
            
            elec_counter +=1
            
            chan_dfs.append(chan_df)
        
chan_dfs = pd.concat(chan_dfs)
chan_dfs['zAge'] =(chan_dfs.age - np.nanmean(chan_dfs.age)) / (2*np.nanstd(chan_dfs.age))
chan_dfs['phit'] = chan_dfs['hits'].astype(int) - 0.5

# # I need to re-integrate the other 40 trials to compute the following: 
# chan_dfs['pCR'] = chan_dfs['correct_rejections'].astype(int) - 0.5


In [ ]:
# formula = 'tfr ~ 0 + phit + zrpe + zpm'
#  'theta', 'alpha', 'beta'
for band in ['hfa', 'theta', 'alpha', 'beta']:
    for region in ['OFC', 'dmPFC', 'HPC', 'AMY', 'ACC']:

        X = ['zrpe', 'zpm', 'phit', 'zAge']
        Intx = ['zrpe:phit', 'zpm:phit']
        y = 'tfr'
        rand_intercept = ['unique_elec_id'] 
        rand_slopes_group = ['unique_elec_id'] 
        rand_slopes_predictor = ['phit']
        categorical = ['phit']
        # formula = make_formula(y=y, X=X, Intx=Intx, 
        #                        rand_effect=rand_intercept, 
        #                        rand_slopes_group=rand_slopes_group, 
        #                        rand_slopes_predictor=rand_slopes_predictor)

        rel_col = ['tfr', 'zrpe', 'zpm', 'phit', 'zAge', 'sex', 'hemi', 'unique_elec_id']

        model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

        # zAge + sex + hemi
#         formula = 'tfr ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1|unique_elec_id)'
        formula = 'tfr ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|unique_elec_id)'
        
        label = f'{region}_{band}_recognition_full_hierarchical_gaussian'

        model, results = run_model(model_df, formula=formula, categorical=categorical,
              priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.9, model_fam='gaussian', 
              output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)
        del results

In [ ]:
# # formula = 'tfr ~ 0 + phit + zrpe + zpm'
# #  'theta', 'alpha', 'beta'
# for band in ['hfa', 'theta', 'alpha', 'beta']:
#     for region in ['dmPFC', 'HPC', 'AMY', 'ACC', 'OFC']:

#         X = ['zrpe', 'zpm', 'pCR', 'zAge']
#         Intx = ['zrpe:pCR', 'zpm:pCR']
#         y = 'tfr'
#         rand_intercept = ['participant', 'unique_elec_id'] 
#         rand_slopes_group = ['participant', 'unique_elec_id'] 
#         rand_slopes_predictor = ['pCR']
#         categorical = ['sex', 'hemi', 'pCR']
#         # formula = make_formula(y=y, X=X, Intx=Intx, 
#         #                        rand_effect=rand_intercept, 
#         #                        rand_slopes_group=rand_slopes_group, 
#         #                        rand_slopes_predictor=rand_slopes_predictor)

#         rel_col = ['tfr', 'zrpe', 'zpm', 'pCR', 'zAge', 'sex', 'hemi', 'unique_elec_id']

#         model_df =  chan_dfs[(chan_dfs.fband==band) & (chan_dfs.region==region)].reset_index(drop=True)[rel_col].dropna()

#         # formula = 'tfr ~ 1+zrpe+zpm+hits+zAge+(1+hits|participant)+(1+zrpe+zpm|unique_elec_id)+zrpe:hits+zpm:hits'
#         formula = 'tfr ~ 1 + zrpe + zpm + pCR + zAge + sex + hemi + zrpe:pCR + zpm:pCR + (1|unique_elec_id)'
#         # +zrpe+zpm+phit+zrpe:phit+zpm:phit
        
#         label = f'{region}_{band}_rejection_retrieval_rand_elec_int_only_t'

#         model, results = run_model(model_df, formula=formula, categorical=categorical,
#               priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
#               output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)


In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
for band in ['theta', 'alpha', 'beta', 'hfa']:
    plot_counter = 0 
    f, ax = plt.subplots(1,5, dpi=300, figsize=(16,4), sharex=True)
    with PdfPages(f'{save_dir_plots}/_recognition_full_hierarchical_gaussian_{band}.pdf') as pdf:    
        for region in ['HPC', 'dmPFC', 'OFC', 'ACC', 'AMY']:
            label = f'{region}_{band}_recognition_full_hierarchical_gaussian_model'
            results = az.from_netcdf(f'{output_dir}/{label}')
            ax[plot_counter].set(xlabel='Weight', ylabel='Parameter')
            # f.suptitle(f'{region}_{band}')
            sns.despine(fig=f, ax=ax[plot_counter], top=True, right=True, left=True, bottom=False)

            X = ['zrpe',
                 'zpm',
                 'phit', 
                 'zrpe:phit',
                 'zpm:phit']
            plot_res(results, X, ax=ax[plot_counter])
            del results
            ax[plot_counter].set(xlabel = 'model weight', title=f'{band}_{region}')
            if plot_counter > 0:
                ax[plot_counter].set(ylabel=None, yticklabels=[], title=f'{band}_{region}')
            plot_counter += 1
        f.tight_layout()
        pdf.savefig(f)


The retrieval results still show: 


1. hippocampal theta decreases before successful recognition 

2. dmPFC, OFC, and AMY hfa decrease before successful recognition of pos RPE images

## Retrieval connectivity

## Amp-Amp coupling

HPC to OFC amp-amp connectivity prior to recognition. 

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
metric = 'amp'
event = 'recog_time'
band1 = 'hfa' 
band2 = 'hfa'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/amp/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_hfa_hfa_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

AMY to OFC amp-amp connectivity prior to recognition. 

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'amp'
event = 'recog_time'
band1 = 'hfa' 
band2 = 'hfa'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/amp/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_hfa_hfa_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

HPC to AMY amp-amp connectivity prior to recognition. 

In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'amp'
event = 'recog_time'
band1 = 'hfa' 
band2 = 'hfa'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/amp/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_hfa_hfa_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

## Coherence

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_retrieval_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_retrieval_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')

In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_retrieval_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5
sns.barplot(data=all_pairs_df, x='good_epoch', y='conn', errorbar='se')


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'coh'

In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t_model'


f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
with PdfPages(f'{save_dir_plots}/retrieval_regression_posteriors_{source_region}_{target_region}_{metric}_retrieval_fully_hierarchical_t.pdf') as pdf:    
    results = az.from_netcdf(f'{output_dir}/{label}')
    ax.set(xlabel='Weight', ylabel='Parameter')
    # f.suptitle(f'{region}_{band}')
    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)

    X = ['zrpe',
         'zpm',
         'phit', 
         'zrpe:phit',
         'zpm:phit']

    plot_res(results, X, ax=ax)
    del results
    ax.set(xlabel = 'model weight', title=f'{source_region}_{target_region}_{metric}')
    f.tight_layout()
    pdf.savefig(f)

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_retrieval_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
source_region = 'HPC'
target_region = 'AMY'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_theta_theta_retrieval_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

## Granger

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
metric = 'granger'
band1 = 'theta'
band2 = 'theta'
event = 'recog_time'

all_pairs_df = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/granger/HPC_OFC_granger_theta_theta_recog_time_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'participant']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|participant)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
source_region = 'AMY'
target_region = 'OFC'
metric = 'granger'
band1 = 'theta'
band2 = 'theta'
event = 'recog_time'

all_pairs_df = pd.read_csv('/sc/arion/projects/guLab/Salman/EphysAnalyses/UI007/scratch/granger/AMY_OFC_granger_theta_theta_recog_time_df')

all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'participant']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|participant)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_theta_theta_retrieval_fully_hierarchical_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

## Retrieval PAC

In [ ]:
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
roi = 'OFC'
event = 'recog_time'
all_dfs = pd.read_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv')


In [ ]:
all_dfs['unique_elec_id'] = np.nan
counter = 0
for participant in all_dfs.participant.unique():
    for label in all_dfs[all_dfs.participant==participant].label.unique():
        all_dfs.loc[(all_dfs.label==label) & (all_dfs.participant==participant), 'unique_elec_id'] = counter
        counter+=1

In [ ]:
all_dfs['phit'] = all_dfs['hits'].astype(int) - 0.5

In [ ]:
region = roi

model_df = all_dfs[all_dfs.good_epoch==1]

rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'unique_elec_id']

model_df =  model_df[rel_col].dropna()



formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|unique_elec_id)'

categorical = ['phit']

label = f'{region}_conn_fully_hierarchical_gaussian'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)
#         del results

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
roi = 'HPC'
event = 'recog_time'
all_dfs = pd.read_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv')


In [ ]:
all_dfs['unique_elec_id'] = np.nan
counter = 0
for participant in all_dfs.participant.unique():
    for label in all_dfs[all_dfs.participant==participant].label.unique():
        all_dfs.loc[(all_dfs.label==label) & (all_dfs.participant==participant), 'unique_elec_id'] = counter
        counter+=1

In [ ]:
all_dfs['phit'] = all_dfs['hits'].astype(int) - 0.5

In [ ]:
region = roi

model_df = all_dfs[all_dfs.good_epoch==1]

rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'unique_elec_id']

model_df =  model_df[rel_col].dropna()



formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|unique_elec_id)'

categorical = ['phit']

label = f'{region}_conn_fully_hierarchical_gaussian'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)
#         del results

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
roi = 'AMY'
event = 'recog_time'
all_dfs = pd.read_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv')


In [ ]:
all_dfs['unique_elec_id'] = np.nan
counter = 0
for participant in all_dfs.participant.unique():
    for label in all_dfs[all_dfs.participant==participant].label.unique():
        all_dfs.loc[(all_dfs.label==label) & (all_dfs.participant==participant), 'unique_elec_id'] = counter
        counter+=1

In [ ]:
all_dfs['phit'] = all_dfs['hits'].astype(int) - 0.5

In [ ]:
region = roi

model_df = all_dfs[all_dfs.good_epoch==1]

rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'unique_elec_id']

model_df =  model_df[rel_col].dropna()



formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1 + zrpe + zpm + phit + zrpe:phit + zpm:phit|unique_elec_id)'

categorical = ['phit']

label = f'{region}_conn_fully_hierarchical_gaussian'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)
#         del results

In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

In [ ]:
region = 'OFC'
band = 'hfa'
label = f'{region}_{band}_retrieval_rand_elec_int_only_t_model'
results = az.from_netcdf(f'{output_dir}/{label}')
az.plot_trace(results)
plt.tight_layout()

In [ ]:
# output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses'

# for region in ['HPC']:
#     for band in ['hfa']:
#         label = f'{region}_{band}_encoding_no_random_elec_model'
#         results = az.from_netcdf(f'{output_dir}/{label}')
#         # construct the model 
        
#         df = chan_dfs[(chan_dfs.region==region) & (chan_dfs.fband==band)].dropna(subset=['tfr', 'zrpe', 'zpm', 'hits'])
        
#         formula = 'tfr ~ 1+zrpe+zpm+hits+(1+zrpe+zpm|participant)+zrpe:hits+zpm:hits'
#         model_fam = 't'
#         priors = None
#         categorical = ['participant']
            
#         model = bmb.Model(formula=formula, 
#                   data=df,
#                  family=model_fam,
#                  priors=priors,
#                  categorical=categorical)
    
#         idata = model.predict(results, data=new_data, inplace=False)
        
#         y_hat = idata.posterior["mpg_mean"]

#         y_hat = results['observed_data']['tfr']
#         y_hat_bounds = az.hdi(y_hat, 0.94)["tfr"].T.to_numpy()
        
#         new_data = pd.DataFrame({"rpe": np.linspace(-2, 2, 100)})
        
#         fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
#         ax.fill_between(new_data["rpe"], y_hat_bounds[0], y_hat_bounds[1], alpha=0.5);

#         f.suptitle(f'{region}_{band}')
#         sns.despine()

First, let's compute the VIF score for predictors to ensure there is not a significant issue of multicollinearity given
the fact that RPE and PM are both related to P(hit=1) 

In [ ]:
predictors = ['zrpe', 'zpm', 'phit']
y = ['tfr']
boi = 'hfa'
roi = 'ACC' 
# sub-select the relevant dataframe 
model_df = chan_dfs[(chan_dfs.fband==boi) & ((chan_dfs.region==roi))].reset_index(drop=True).dropna(subset=y+predictors)
predictor_df = model_df[predictors]
# predictor_df['zrpe:phit'] = predictor_df['zrpe'] * predictor_df['phit']
# predictor_df['zpm:phit'] = predictor_df['zpm'] * predictor_df['phit']
X = add_constant(predictor_df)
pd.Series([variance_inflation_factor(X.values, i) 
               for i in range(X.shape[1])], 
              index=X.columns)

## PAC Regression

9/12/2024: Can I load different subsets of the PAC data to get significant PAC? 

In [ ]:
save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'

roi = 'OFC'
event = 'feedback_start'
all_dfs = pd.read_csv(f'{save_dir_data}/all_{roi}_pac_{event}.csv')


all_dfs['unique_elec_id'] = np.nan
counter = 0
for participant in all_dfs.participant.unique():
    for label in all_dfs[all_dfs.participant==participant].label.unique():
        all_dfs.loc[(all_dfs.label==label) & (all_dfs.participant==participant), 'unique_elec_id'] = counter
        counter+=1
        
all_dfs.hemi = all_dfs.hemi.str.lower()

In [ ]:
all_dfs

In [ ]:
rel_col = ['avg_conn', 'sum_conn', 'hemi', 'zrpe', 'zpm', 'unique_elec_id', 'participant']
model_df =  all_dfs[rel_col].dropna()

formula = 'avg_conn ~ 1 + zrpe + zpm + hemi + (1|participant) + (1 + zrpe + zpm|unique_elec_id)'

categorical = ['participant', 'unique_elec_id', 'hemi']

label = f'{roi}_avgpac_fully_hierarchical_gaussian'

model_rpe, results_rpe = run_model(model_df, formula=formula, categorical=categorical,
                           priors=None, cores=4, chains=4, tune=4000, draws=2000, target_accept=0.85, model_fam='gaussian', 
                           output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', 
                           return_model=True, save_model_res=True, label=label, sampler='mcmc')
        

In [ ]:
az.plot_trace(results_rpe, var_names=['hemi'])

In [ ]:
output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels'
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4), sharex=True)
with PdfPages(f'{save_dir_plots}/conn_regression_posteriors_{roi}_encoding_fully_hierarchical_gaussian.pdf') as pdf:    
    label = f'{roi}_pac_fully_hierarchical_gaussian'
#     results = az.from_netcdf(f'{output_dir}/{label}')
    ax.set(xlabel='Weight', ylabel='Parameter')

    sns.despine(fig=f, ax=ax, top=True, right=True, left=True, bottom=False)

    X = ['zrpe',
         'zpm']
    
    plot_res(results_rpe, X, ax=ax)
    ax.set(xlabel = 'model weight', title=f'{roi}')
    f.tight_layout()
    pdf.savefig(f)

## Retrieval amp connectivity

First, let's replicate the same HPC-OFC amplitude connectivity between HFA and theta

In [ ]:
source_region = 'HPC'
target_region = 'OFC'
metric = 'amp'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_retrieval_df')
all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_recognition_rand_int_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

Now let's look at dmPFC - OFC theta amp-amp connectivity:

In [ ]:
source_region = 'dmPFC'
target_region = 'OFC'
metric = 'amp'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_retrieval_df')
all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_recognition_rand_int_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

Now let's do the same for theta coherence:

In [ ]:
source_region = 'dmPFC'
target_region = 'OFC'
metric = 'coh'

all_pairs_df = pd.read_csv(f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/{source_region}_{target_region}_{metric}_retrieval_df')
all_pairs_df['phit'] = all_pairs_df['hits'].astype(int) - 0.5


In [ ]:
rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'pair_label']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1|pair_label)'

categorical = ['phit']

label = f'{source_region}_{target_region}_{metric}_recognition_rand_int_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)

## Retrieval OFC PAC 

In [ ]:
roi = 'OFC'
event = 'feedback_start' 
metric = 'pac'


all_pairs_df = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes/all_{roi}_pac_{event}.csv')
# add elec_id
elec_counter = 0 
for participant in all_pairs_df.participant.unique():
    for label in all_pairs_df[all_pairs_df.participant==participant].label.unique():
        all_pairs_df.loc[(all_pairs_df.participant==participant) & (all_pairs_df.label==label), 'unique_elec_id'] = elec_counter
        elec_counter +=1


In [ ]:
sns.lmplot(data=all_pairs_df, x='avg_conn', y='rpe', hue = 'unique_elec_id', ci=None, scatter= False)




In [ ]:
roi = 'dmPFC'
event = 'feedback_start' 
metric = 'pac'


all_pairs_df1 = pd.read_csv(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes/all_{roi}_pac_{event}.csv')
# add elec_id
elec_counter = 0 
for participant in all_pairs_df1.participant.unique():
    for label in all_pairs_df1[all_pairs_df1.participant==participant].label.unique():
        all_pairs_df1.loc[(all_pairs_df1.participant==participant) & (all_pairs_df1.label==label), 'unique_elec_id'] = elec_counter
        elec_counter +=1


In [ ]:
sns.regplot(x=all_pairs_df.groupby('participant').mean(numeric_only=True)['sum_conn'], 
            y=all_pairs_df1[all_pairs_df1.participant.isin(all_pairs_df.groupby('participant').mean(numeric_only=True)['sum_conn'].index)].groupby('participant').mean(numeric_only=True)['sum_conn'])


In [ ]:
pearsonr(all_pairs_df.groupby('participant').mean(numeric_only=True)['avg_conn'],
         all_pairs_df1[all_pairs_df1.participant.isin(all_pairs_df.groupby('participant').mean(numeric_only=True)['avg_conn'].index)].groupby('participant').mean(numeric_only=True)['avg_conn'])
         

OFC and dmPFC PAC vary across subjects, but are well correlated with one another, and inversely correlated with learning rate.  

In [ ]:
all_pairs_df1[all_pairs_df1.participant.isin(all_pairs_df.groupby('participant').mean(numeric_only=True)['avg_conn'].index)].groupby('participant').mean(numeric_only=True)['avg_conn']



In [ ]:
sns.barplot(data=all_pairs_df, x='participant', y='avg_conn', errorbar='se')

In [ ]:
### rel_col = ['conn', 'zrpe', 'zpm', 'phit', 'unique_elec_id']
model_df = all_pairs_df[all_pairs_df.good_epoch==1][rel_col].dropna()

formula = 'conn ~ 1 + zrpe + zpm + phit + zrpe:phit + zpm:phit + (1|unique_elec_id)'

categorical = ['phit']

label = f'{roi}_{metric}_recognition_rand_int_t'

model, results = run_model(model_df, formula=formula, categorical=categorical,
      priors=None, cores=4, chains=4, tune=4000, draws=4000, target_accept=0.8, model_fam='t', 
      output_dir='/sc/arion/projects/guLab/Salman/EphysAnalyses/BayesianRegressionModels', return_model=True, save_model_res = True, label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
ax.set(xlabel='Weight', ylabel='Parameter')
            
X = ['zrpe',
     'zpm',
     'phit', 
     'zrpe:phit',
     'zpm:phit']
plot_res(results, X, ax=ax)